# HCG-SelectiveECG — version Ronald / carpeta compartida actual

Este notebook implementa la nueva aplicacion del proyecto:

**Human-Certainty-Guided Selective ECG Classification (HCG-SelectiveECG)**.

Esta version esta adaptada para ejecutarse en la **misma carpeta compartida** donde Ronald ejecuta:

`ptbxl_cnn1d_hard_soft_training_colab_ronald.ipynb`

Es decir, espera una raiz tipo:

```text
ptbxl_likelihood_calibration/
├── data/
│   ├── raw/
│   │   └── ptbxl_base/
│   │       ├── ptbxl_database.csv
│   │       ├── scp_statements.csv
│   │       └── records100/
│   └── processed/
│       ├── ptbxl_compatible_master.csv
│       ├── labels_hard.csv
│       ├── labels_soft.csv
│       └── label_audit.csv
```

El notebook **no sobreescribe** el experimento anterior. Guarda salidas en:

```text
ptbxl_likelihood_calibration/runs_hcg_selective/RUN_ID/
ptbxl_likelihood_calibration/deliverables_hcg_selective/
ptbxl_likelihood_calibration/data/processed_hcg_selective/
```

Modelos:

- `B1_HARD_DIAG`: clasificador diagnostico hard-label.
- `B2_SOFT_DIRECT`: baseline soft-label directo.
- `B3_THRESHOLD_UNCERTAINTY`: baseline de derivacion por cercania al umbral de B1.
- `P0_MTL_UNBALANCED`: multi-task sin balanceo.
- `P1_MTL_BALANCED_PROPOSED`: metodo principal, multi-task balanceado por bins de ambiguedad.

Si Colab se cae, conservar el `RUN_ID` impreso, pegarlo en `RUN_ID_OVERRIDE` y reejecutar con `RESUME=True`.


## Protocolo multi-semilla (robustez estadistica)

Para reportar resultados en el paper, correr el notebook una vez por cada
semilla de `SEED_LIST` (celda 3), cambiando `SEED_INDEX = 0,1,2,3,4` y
ejecutando de arriba a abajo. Cada semilla guarda en `runs_hcg_selective/`
bajo un `RUN_ID` estable `hcg_selective_ecg_full_seedNN`.

La semilla 42 ya fue ejecutada; la celda final de **agregacion multi-semilla**
la reutiliza automaticamente, asi que solo hace falta correr las 4 restantes
(`SEED_INDEX = 1,2,3,4`). Tras la ultima, la agregacion escribe media +/- DE
e intervalos de confianza entre semillas en
`runs_hcg_selective/_aggregate_multiseed/`.


In [ ]:
# ============================================================
# 1. Instalacion de dependencias
# ============================================================
import sys, subprocess, importlib.util, os

def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])

for import_name, pip_name in [
    ("wfdb", "wfdb"),
    ("sklearn", "scikit-learn"),
    ("tqdm", "tqdm"),
    ("scipy", "scipy"),
    ("seaborn", "seaborn"),
]:
    ensure_package(import_name, pip_name)

try:
    import pyarrow  # noqa
except Exception:
    # Parquet es deseable, pero CSV queda como fallback.
    pass

print("Dependencias base listas.")


In [ ]:
# ============================================================
# 2. Imports, Drive, dispositivo y semilla
# ============================================================
from pathlib import Path
from datetime import datetime
import ast
import gc
import json
import math
import os
import random
import shutil
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from scipy.stats import spearmanr

import wfdb

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    brier_score_loss,
    accuracy_score,
    mean_absolute_error,
    mean_squared_error,
)

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print("DEVICE:", DEVICE, "|", GPU_NAME)


In [ ]:
# ============================================================
# 3. Configuracion global PORTABLE estilo notebook Ronald
# ============================================================
# Esta version viene en full para que Ronald pueda correrla directamente.
# Para una prueba rapida, cambiar a "debug".
RUN_MODE = "full"   # "debug" o "full"

RUN_ID_OVERRIDE = ""
RESUME = True
SKIP_COMPLETED = True
FORCE_RETRAIN = False
FORCE_PREPROCESS = False

# Si la carpeta compartida tiene otro nombre o ubicacion, poner ruta completa.
PROJECT_ROOT_OVERRIDE = ""  # opcional: ruta local o de Drive a ptbxl_likelihood_calibration

PREFERRED_PROJECT_FOLDER_NAMES = [
    "ptbxl_likelihood_calibration",
    "ptbxl_likelihood_calibration_shared",
    "ptbxl_likelihood_calibration_portable",
]

def es_raiz_proyecto_valida(path: Path) -> bool:
    return (
        path.exists()
        and (path / "data" / "raw" / "ptbxl_base" / "records100").exists()
        and (path / "data" / "processed").exists()
    )

def resolver_project_root() -> Path:
    if PROJECT_ROOT_OVERRIDE.strip():
        root = Path(PROJECT_ROOT_OVERRIDE.strip())
        if not es_raiz_proyecto_valida(root):
            raise FileNotFoundError(
                f"PROJECT_ROOT_OVERRIDE no parece raiz valida: {root}\n"
                "Debe contener data/raw/ptbxl_base/records100 y data/processed."
            )
        return root

    candidates = []
    mydrive = Path("/content/drive/MyDrive")
    for name in PREFERRED_PROJECT_FOLDER_NAMES:
        candidates.append(mydrive / name)

    if mydrive.exists():
        try:
            for p in mydrive.iterdir():
                if p.is_dir() and "ptbxl_likelihood" in p.name.lower():
                    candidates.append(p)
        except Exception as e:
            print("No se pudo escanear MyDrive:", repr(e))

    shared_drives_root = Path("/content/drive/Shareddrives")
    if shared_drives_root.exists():
        try:
            for shared_drive in shared_drives_root.iterdir():
                if shared_drive.is_dir():
                    for name in PREFERRED_PROJECT_FOLDER_NAMES:
                        candidates.append(shared_drive / name)
        except Exception as e:
            print("No se pudo escanear Shared drives:", repr(e))

    seen, unique = set(), []
    for c in candidates:
        s = str(c)
        if s not in seen:
            unique.append(c)
            seen.add(s)

    for c in unique:
        if es_raiz_proyecto_valida(c):
            return c

    print("Candidatos revisados:")
    for c in unique:
        print(" -", c, "exists=", c.exists())
    raise FileNotFoundError(
        "No encontre la carpeta raiz del proyecto. Agrega la carpeta compartida "
        "como acceso directo a Mi unidad o define PROJECT_ROOT_OVERRIDE."
    )

ROOT = resolver_project_root()
PROJECT_ROOT = ROOT

DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
RAW_BASE = RAW_DIR / "ptbxl_base"
RECORDS100_PATH = RAW_BASE / "records100"
PTBXL_DB_PATH = RAW_BASE / "ptbxl_database.csv"
SCP_PATH = RAW_BASE / "scp_statements.csv"
BASE_PROJECT_CANDIDATES = [ROOT]
RAW_BASE_CANDIDATES = [RAW_BASE]

INPUT_PROCESSED_DIR = DATA_DIR / "processed"
MASTER_PATH = INPUT_PROCESSED_DIR / "ptbxl_compatible_master.csv"
LABELS_HARD_PATH = INPUT_PROCESSED_DIR / "labels_hard.csv"
LABELS_SOFT_PATH = INPUT_PROCESSED_DIR / "labels_soft.csv"
LABEL_AUDIT_PATH = INPUT_PROCESSED_DIR / "label_audit.csv"

# Cache propio de este nuevo experimento; no toca data/processed anterior.
PROCESSED_DIR = DATA_DIR / "processed_hcg_selective"

RUNS_DIR = ROOT / "runs_hcg_selective"
DELIVERABLES_DIR = ROOT / "deliverables_hcg_selective"
COMUNICACION_DIR = ROOT / "comunicacion"
LOCAL_RECORDS100 = Path("/content/ptbxl_records100_local")
LOCAL_CACHE_DIR = Path("/content/ptbxl_cache_records100")

CLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]
ETA_REVIEW = 0.20
Q_VALUES = [0.05, 0.10, 0.20, 0.30]

# ------------------------------------------------------------
# Multi-semilla para robustez estadistica (paper).
# Correr el notebook UNA VEZ por cada indice de SEED_LIST,
# cambiando SEED_INDEX = 0,1,2,3,4 y ejecutando todo de arriba a abajo.
# Cada semilla escribe en su propio RUN_DIR estable (..._seedNN),
# de modo que RESUME/SKIP_COMPLETED funcionan por-semilla y reanudan.
# La celda de agregacion final combina todas las semillas (media +/- DE).
# NOTA: la semilla 42 ya fue corrida en un RUN_ID con timestamp; la celda
# de agregacion la reutiliza automaticamente, no hace falta re-correrla.
# ------------------------------------------------------------
SEED_LIST = [42, 13, 123, 2024, 7]
SEED_INDEX = 4                      # <-- cambiar a 1,2,3,4 en cada nueva corrida
SEED = SEED_LIST[SEED_INDEX]

# Re-sembrar TODO aqui: la celda 2 sembro con 42; este es el punto autoritativo
# para que cambiar SEED cambie realmente la inicializacion de pesos y el shuffle.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Aceleracion I/O igual al notebook de Ronald.
USE_LOCAL_RECORDS100 = bool(IN_COLAB)
CACHE_SIGNALS_LOCAL = False

if RUN_MODE == "debug":
    DEBUG_N_TRAIN = 1024
    DEBUG_N_VAL = 256
    DEBUG_N_TEST = 256
    MAX_EPOCHS = 2
    BATCH_SIZE = 64
    N_BOOTSTRAP = 100
    LEAD_ABLATION_N = 64
    NOISE_PERTURB_N = 64
else:
    DEBUG_N_TRAIN = None
    DEBUG_N_VAL = None
    DEBUG_N_TEST = None
    # Full conservador para que corra en Colab T4/local sin bloquear demasiado.
    MAX_EPOCHS = 40
    BATCH_SIZE = 128
    N_BOOTSTRAP = 1000
    LEAD_ABLATION_N = 500
    NOISE_PERTURB_N = 500

LR = 1e-3
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 8 if RUN_MODE == "full" else 2
REVIEW_LOSS_WEIGHT = 0.5
WARMUP_EPOCHS_MTL = 1 if RUN_MODE == "debug" else 5

CPU_COUNT = os.cpu_count() or 2
if IN_COLAB:
    NUM_WORKERS = min(4, max(2, CPU_COUNT - 1))
else:
    NUM_WORKERS = 0
PIN_MEMORY = bool(DEVICE.type == "cuda")
PERSISTENT_WORKERS = bool(NUM_WORKERS > 0)
PREFETCH_FACTOR = 4 if NUM_WORKERS > 0 else None
MIXED_PRECISION = bool(DEVICE.type == "cuda")
AMP_DTYPE = torch.float16
USE_TORCH_COMPILE = False

if RUN_ID_OVERRIDE.strip():
    RUN_ID = RUN_ID_OVERRIDE.strip()
else:
    # Nombre estable por-semilla: agregacion reproducible y RESUME idempotente.
    RUN_ID = f"hcg_selective_ecg_{RUN_MODE}_seed{SEED}"

RUN_DIR = RUNS_DIR / RUN_ID
CONFIG_DIR = RUN_DIR / "config"
MODELS_DIR = RUN_DIR / "models"
RESULTS_DIR = RUN_DIR / "results"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURE_DATA_DIR = RESULTS_DIR / "figure_data"
FIGURES_DIR = RESULTS_DIR / "figures"
EXPLAIN_DIR = RESULTS_DIR / "explainability"
ATTR_DIR = EXPLAIN_DIR / "attributions"
MANIFEST_DIR = RESULTS_DIR / "manifests"
LOGS_DIR = RUN_DIR / "logs"
NOTEBOOKS_DIR = ROOT
SRC_DIR = ROOT / "src_hcg_selective"

for d in [
    PROCESSED_DIR, RUNS_DIR, DELIVERABLES_DIR, RUN_DIR, CONFIG_DIR, MODELS_DIR,
    TABLES_DIR, FIGURE_DATA_DIR, FIGURES_DIR, EXPLAIN_DIR, ATTR_DIR,
    MANIFEST_DIR, LOGS_DIR, SRC_DIR, COMUNICACION_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT / ROOT:", ROOT)
print("RUN_ID:", RUN_ID)
print("RUN_DIR:", RUN_DIR)
print("DELIVERABLES_DIR:", DELIVERABLES_DIR)
print("RUN_MODE:", RUN_MODE)
print("MAX_EPOCHS:", MAX_EPOCHS)
print("BATCH_SIZE:", BATCH_SIZE)
print("DEVICE:", DEVICE, "|", GPU_NAME)
print("NUM_WORKERS:", NUM_WORKERS)
print("PIN_MEMORY:", PIN_MEMORY)
print("PERSISTENT_WORKERS:", PERSISTENT_WORKERS)
print("PREFETCH_FACTOR:", PREFETCH_FACTOR)
print("MIXED_PRECISION:", MIXED_PRECISION)


In [ ]:
# ============================================================
# 4. Logging y utilidades de archivo
# ============================================================
LOG_PATH = LOGS_DIR / "run.log"

def log(msg):
    text = f"[{datetime.now().isoformat(timespec='seconds')}] {msg}"
    print(text)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(text + "\n")

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def load_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_dataframe(df, path, index=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=index)

def save_parquet_or_csv(df, parquet_path):
    parquet_path = Path(parquet_path)
    parquet_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        df.to_parquet(parquet_path, index=False)
        return parquet_path
    except Exception as e:
        csv_path = parquet_path.with_suffix(".csv")
        df.to_csv(csv_path, index=False)
        log(f"Parquet no disponible; se guardo CSV fallback: {csv_path}. Motivo: {e}")
        return csv_path

def package_versions():
    versions = {}
    for name in ["numpy", "pandas", "torch", "wfdb", "sklearn", "scipy", "matplotlib", "seaborn"]:
        try:
            mod = __import__(name)
            versions[name] = getattr(mod, "__version__", "unknown")
        except Exception:
            versions[name] = "not_installed"
    return versions

save_json(package_versions(), LOGS_DIR / "package_versions.json")
with open(LOGS_DIR / "environment.txt", "w", encoding="utf-8") as f:
    f.write(f"RUN_ID={RUN_ID}\nRUN_MODE={RUN_MODE}\nDEVICE={DEVICE}\nGPU={GPU_NAME}\n")

main_config = {
    "run_id": RUN_ID,
    "run_mode": RUN_MODE,
    "project_root": str(PROJECT_ROOT),
    "classes": CLASSES,
    "eta_review": ETA_REVIEW,
    "q_values": Q_VALUES,
    "seed": SEED,
    "max_epochs": MAX_EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "mixed_precision": MIXED_PRECISION,
}
save_json(main_config, CONFIG_DIR / "main_config.json")
log("Configuracion inicial guardada.")


In [ ]:
# ============================================================
# 4b. Contexto metodologico persistente en Drive
# ============================================================
context_text = '''# 00_CONTEXT_HCG_SELECTIVE_ECG

## Punto de vista cientifico

Este notebook reconstruye el proyecto como clasificacion selectiva de ECG guiada
por certeza humana. El objetivo no es maximizar accuracy diagnostica ni afirmar
que los soft labels directos son superiores. La pregunta central es si los
likelihoods diagnosticos de PTB-XL sirven como senal auxiliar para identificar
ECG que requieren revision medica cuidadosa.

## Metodo

Nombre largo: Human-Certainty-Guided Selective ECG Classification.
Nombre corto: HCG-SelectiveECG.

Se conserva la clasificacion multi-label NORM, MI, STTC, CD, HYP, pero se
agrega una cabeza de revision entrenada con ambiguedad humana:

`a_i = max_{k positivo}(1 - likelihood_k)`.

La decision operativa final es: diagnostico sugerido + aceptar/revision.
'''
(PROJECT_ROOT / "00_CONTEXT_HCG_SELECTIVE_ECG.md").write_text(context_text, encoding="utf-8")
save_json({
    "method": "HCG-SelectiveECG",
    "run_id": RUN_ID,
    "project_root": str(PROJECT_ROOT),
    "data_source_candidates": [str(p) for p in BASE_PROJECT_CANDIDATES],
    "raw_base_candidates": [str(p) for p in RAW_BASE_CANDIDATES],
    "classes": CLASSES,
    "eta_review": ETA_REVIEW,
    "models": ["B1_HARD_DIAG", "B2_SOFT_DIRECT", "B3_THRESHOLD_UNCERTAINTY", "P0_MTL_UNBALANCED", "P1_MTL_BALANCED_PROPOSED"],
    "created_at": datetime.now().isoformat(timespec="seconds"),
}, PROJECT_ROOT / "codex_context_hcg_selective_review.json")
log("Contexto metodologico guardado en PROJECT_ROOT.")


In [ ]:
# ============================================================
# 5. Verificacion de rutas de la carpeta compartida
# ============================================================
required_paths = {
    "ROOT": ROOT,
    "RECORDS100_PATH": RECORDS100_PATH,
    "INPUT_PROCESSED_DIR": INPUT_PROCESSED_DIR,
    "MASTER_PATH": MASTER_PATH,
    "LABELS_HARD_PATH": LABELS_HARD_PATH,
    "LABELS_SOFT_PATH": LABELS_SOFT_PATH,
}

path_rows = []
for name, p in required_paths.items():
    exists = p.exists()
    if exists and p.is_file():
        size = p.stat().st_size
    elif exists and p.is_dir():
        try:
            size = len(list(p.iterdir()))
        except Exception:
            size = None
    else:
        size = None
    path_rows.append({"item": name, "path": str(p), "exists": bool(exists), "size_or_count": size})

path_check_df = pd.DataFrame(path_rows)
display(path_check_df)
path_check_df.to_csv(MANIFEST_DIR / "path_checks.csv", index=False)

missing = [r["item"] for r in path_rows if not r["exists"]]
if missing:
    raise FileNotFoundError(
        "Faltan rutas esenciales en la carpeta compartida de Ronald: "
        + ", ".join(missing)
        + ". Debe ejecutar este notebook en la misma carpeta del notebook hard/soft."
    )

if not PTBXL_DB_PATH.exists() or not SCP_PATH.exists():
    print("Aviso: no se encontraron ptbxl_database.csv/scp_statements.csv en raw.")
    print("No es critico: se usaran ptbxl_compatible_master.csv, labels_hard.csv y labels_soft.csv.")
else:
    print("Metadata raw PTB-XL disponible:", RAW_BASE)

save_json({
    "root": str(ROOT),
    "records100": str(RECORDS100_PATH),
    "master_path": str(MASTER_PATH),
    "labels_hard_path": str(LABELS_HARD_PATH),
    "labels_soft_path": str(LABELS_SOFT_PATH),
    "run_dir": str(RUN_DIR),
}, CONFIG_DIR / "resolved_paths.json")


In [ ]:
# ============================================================
# 6. Carga desde data/processed de Ronald y construccion HCG
# ============================================================
def leer_csv_seguro(path: Path, nombre: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"No existe {nombre}: {path}")
    df = pd.read_csv(path)
    print(f"{nombre}: {df.shape} -> {path}")
    return df

def normalizar_ecg_id_col(df: pd.DataFrame) -> pd.DataFrame:
    cols_lower = {c.lower(): c for c in df.columns}
    if "ecg_id" in cols_lower and cols_lower["ecg_id"] != "ecg_id":
        df = df.rename(columns={cols_lower["ecg_id"]: "ecg_id"})
    return df

def buscar_columna(df: pd.DataFrame, candidatos) -> str:
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidatos:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None

def encontrar_col_label(df: pd.DataFrame, clase: str, tipo: str):
    exactos = [clase, clase.lower(), f"{clase}_{tipo}", f"{clase.lower()}_{tipo}", f"{tipo}_{clase}", f"{tipo}_{clase.lower()}"]
    lower_map = {c.lower(): c for c in df.columns}
    for e in exactos:
        if e.lower() in lower_map:
            return lower_map[e.lower()]
    for c in df.columns:
        cl = c.lower()
        if clase.lower() in cl and tipo.lower() in cl:
            return c
    for c in df.columns:
        if c.upper() == clase.upper():
            return c
    return None

def cargar_labels(path: Path, tipo: str, master_df: pd.DataFrame) -> pd.DataFrame:
    df = leer_csv_seguro(path, f"labels_{tipo}")
    df = normalizar_ecg_id_col(df)
    if "ecg_id" not in df.columns:
        if len(df) == len(master_df) and "ecg_id" in master_df.columns:
            df.insert(0, "ecg_id", master_df["ecg_id"].values)
        else:
            raise ValueError(f"labels_{tipo} no tiene ecg_id y no se puede alinear por longitud.")
    cols, rename = [], {}
    for clase in CLASSES:
        col = encontrar_col_label(df, clase, tipo)
        if col is None:
            raise ValueError(f"No encontre columna para {clase} en labels_{tipo}.")
        cols.append(col)
        rename[col] = clase
    out = df[["ecg_id"] + cols].rename(columns=rename).copy()
    out["ecg_id"] = out["ecg_id"].astype(int)
    for c in CLASSES:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0)
    return out.set_index("ecg_id").sort_index()

def construir_ambiguedad_desde_labels(master_df, labels_hard, labels_soft):
    ids = master_df["ecg_id"].astype(int).values
    y_hard = labels_hard.loc[ids, CLASSES].values.astype(np.float32)
    y_soft = labels_soft.loc[ids, CLASSES].values.astype(np.float32)
    y_soft = np.clip(y_soft, 0.0, 1.0)

    ambiguity_score = np.zeros(len(master_df), dtype=np.float32)
    for i in range(len(master_df)):
        positives = np.where(y_hard[i] == 1)[0]
        ambiguity_score[i] = 0.0 if len(positives) == 0 else float(np.max(1.0 - y_soft[i, positives]))

    review_binary = (ambiguity_score >= ETA_REVIEW).astype(np.float32)
    ambiguity_bins = np.full(len(master_df), "B0", dtype=object)
    ambiguity_bins[(ambiguity_score > 0.0) & (ambiguity_score <= 0.2)] = "B1"
    ambiguity_bins[(ambiguity_score > 0.2) & (ambiguity_score <= 0.5)] = "B2"
    ambiguity_bins[ambiguity_score > 0.5] = "B3"

    label_audit = pd.DataFrame({
        "ecg_id": ids,
        "positive_classes": [
            ",".join([CLASSES[k] for k in np.where(y_hard[i] == 1)[0]])
            for i in range(len(master_df))
        ],
        "ambiguity_score": ambiguity_score,
        "review_binary": review_binary.astype(int),
        "ambiguity_bin": ambiguity_bins,
    })
    return y_hard, y_soft, ambiguity_score, review_binary, ambiguity_bins, label_audit

def load_or_build_processed():
    needed = [
        PROCESSED_DIR / "labels_hard.npy",
        PROCESSED_DIR / "labels_soft_likelihood.npy",
        PROCESSED_DIR / "ambiguity_score.npy",
        PROCESSED_DIR / "review_binary.npy",
        PROCESSED_DIR / "ambiguity_bins.npy",
        PROCESSED_DIR / "metadata_splits.csv",
    ]
    if all(p.exists() for p in needed) and not FORCE_PREPROCESS:
        log("Cargando cache HCG desde data/processed_hcg_selective.")
        master_df = pd.read_csv(PROCESSED_DIR / "metadata_splits.csv")
        return (
            master_df,
            np.load(PROCESSED_DIR / "labels_hard.npy"),
            np.load(PROCESSED_DIR / "labels_soft_likelihood.npy"),
            np.load(PROCESSED_DIR / "ambiguity_score.npy"),
            np.load(PROCESSED_DIR / "review_binary.npy"),
            np.load(PROCESSED_DIR / "ambiguity_bins.npy", allow_pickle=True),
        )

    master_df = leer_csv_seguro(MASTER_PATH, "ptbxl_compatible_master")
    master_df = normalizar_ecg_id_col(master_df)
    if "ecg_id" not in master_df.columns:
        raise ValueError("ptbxl_compatible_master.csv no contiene ecg_id.")
    master_df["ecg_id"] = master_df["ecg_id"].astype(int)

    labels_hard = cargar_labels(LABELS_HARD_PATH, "hard", master_df)
    labels_soft = cargar_labels(LABELS_SOFT_PATH, "soft", master_df)

    ids_master = set(master_df["ecg_id"].astype(int))
    ids_labels = set(labels_hard.index.astype(int)) & set(labels_soft.index.astype(int))
    ids_comunes = sorted(ids_master & ids_labels)
    if len(ids_comunes) != len(master_df):
        print(f"Advertencia: master tiene {len(master_df)} filas, ids comunes labels={len(ids_comunes)}")
    master_df = master_df[master_df["ecg_id"].isin(ids_comunes)].copy().reset_index(drop=True)
    labels_hard = labels_hard.loc[master_df["ecg_id"].astype(int)]
    labels_soft = labels_soft.loc[master_df["ecg_id"].astype(int)]

    fold_col = buscar_columna(master_df, ["strat_fold", "fold", "cv_fold"])
    split_col = buscar_columna(master_df, ["split", "dataset_split", "partition", "_split"])
    if fold_col is not None:
        master_df["split"] = np.where(master_df[fold_col].isin([1,2,3,4,5,6,7,8]), "train",
                               np.where(master_df[fold_col].eq(9), "val",
                               np.where(master_df[fold_col].eq(10), "test", "unused")))
    elif split_col is not None:
        master_df["split"] = master_df[split_col].astype(str).str.lower().replace({"validation": "val", "valid": "val"})
    else:
        raise ValueError("No se encontro strat_fold/fold/split para separar train/val/test.")

    y_hard, y_soft, ambiguity_score, review_binary, ambiguity_bins, label_audit = construir_ambiguedad_desde_labels(master_df, labels_hard, labels_soft)
    master_df["ambiguity_score"] = ambiguity_score
    master_df["review_binary"] = review_binary.astype(int)
    master_df["ambiguity_bin"] = ambiguity_bins

    save_parquet_or_csv(master_df, PROCESSED_DIR / "metadata_splits.parquet")
    master_df.to_csv(PROCESSED_DIR / "metadata_splits.csv", index=False)
    np.save(PROCESSED_DIR / "labels_hard.npy", y_hard)
    np.save(PROCESSED_DIR / "labels_soft_likelihood.npy", y_soft)
    np.save(PROCESSED_DIR / "ambiguity_score.npy", ambiguity_score)
    np.save(PROCESSED_DIR / "review_binary.npy", review_binary)
    np.save(PROCESSED_DIR / "ambiguity_bins.npy", ambiguity_bins)
    save_json(CLASSES, PROCESSED_DIR / "class_names.json")
    label_audit.to_csv(PROCESSED_DIR / "label_audit_hcg.csv", index=False)

    amb_cols = [c for c in ["ecg_id", "strat_fold", "age", "sex", "split", "ambiguity_score", "review_binary", "ambiguity_bin"] if c in master_df.columns]
    master_df[amb_cols].to_csv(PROCESSED_DIR / "ambiguity_audit.csv", index=False)
    log("Cache HCG creado desde data/processed de Ronald.")
    return master_df, y_hard, y_soft, ambiguity_score, review_binary, ambiguity_bins

master, Y_HARD, Y_SOFT, AMBIGUITY, REVIEW_BINARY, AMBIGUITY_BINS = load_or_build_processed()
print(master.shape, Y_HARD.shape, Y_SOFT.shape)


In [ ]:
# ============================================================
# 7. Auditoria del dataset y pesos de entrenamiento
# ============================================================
def split_indices(master, split):
    return np.where(master["split"].values == split)[0]

idx_train_full = split_indices(master, "train")
idx_val_full = split_indices(master, "val")
idx_test_full = split_indices(master, "test")

if RUN_MODE == "debug":
    rng = np.random.default_rng(SEED)
    idx_train = rng.choice(idx_train_full, size=min(DEBUG_N_TRAIN, len(idx_train_full)), replace=False)
    idx_val = rng.choice(idx_val_full, size=min(DEBUG_N_VAL, len(idx_val_full)), replace=False)
    idx_test = rng.choice(idx_test_full, size=min(DEBUG_N_TEST, len(idx_test_full)), replace=False)
else:
    idx_train, idx_val, idx_test = idx_train_full, idx_val_full, idx_test_full

SPLIT_INDICES = {"train": idx_train, "val": idx_val, "test": idx_test}

def compute_pos_weights(y_train):
    pos = y_train.sum(axis=0)
    neg = y_train.shape[0] - pos
    weights = np.sqrt(neg / np.maximum(pos, 1))
    weights = np.clip(weights, 1.0, 10.0)
    return weights.astype(np.float32)

def compute_bin_weights(bins_train):
    labels = ["B0", "B1", "B2", "B3"]
    n = len(bins_train)
    B = len(labels)
    weights = {}
    for b in labels:
        nb = int(np.sum(bins_train == b))
        if nb == 0:
            weights[b] = 1.0
        else:
            weights[b] = float(np.clip(np.sqrt(n / (B * nb)), 0.5, 5.0))
    return weights

DIAG_POS_WEIGHTS = compute_pos_weights(Y_HARD[idx_train])
BIN_WEIGHTS = compute_bin_weights(AMBIGUITY_BINS[idx_train])
save_json(dict(zip(CLASSES, DIAG_POS_WEIGHTS.tolist())), CONFIG_DIR / "diagnostic_pos_weights.json")
save_json(BIN_WEIGHTS, CONFIG_DIR / "ambiguity_bin_weights.json")

def dataset_audit_table():
    rows = []
    for split, idx in SPLIT_INDICES.items():
        row = {
            "split": split,
            "N": len(idx),
            "N_patients": master.iloc[idx]["patient_id"].nunique() if "patient_id" in master.columns else np.nan,
            "review_binary_N": int(REVIEW_BINARY[idx].sum()),
            "ambig_mean": float(np.mean(AMBIGUITY[idx])),
            "ambig_median": float(np.median(AMBIGUITY[idx])),
        }
        for k, cls in enumerate(CLASSES):
            row[f"prev_{cls}"] = float(Y_HARD[idx, k].mean())
            row[f"N_{cls}"] = int(Y_HARD[idx, k].sum())
        for b in ["B0", "B1", "B2", "B3"]:
            row[f"N_{b}"] = int(np.sum(AMBIGUITY_BINS[idx] == b))
        rows.append(row)
    return pd.DataFrame(rows)

dataset_audit = dataset_audit_table()
dataset_audit.to_csv(TABLES_DIR / "dataset_audit.csv", index=False)
dataset_audit.to_csv(FIGURE_DATA_DIR / "fig02_likelihood_distribution.csv", index=False)
display(dataset_audit)

sanity = {
    "n_total": int(len(master)),
    "n_train": int(len(idx_train)),
    "n_val": int(len(idx_val)),
    "n_test": int(len(idx_test)),
    "hard_binary": bool(np.isin(Y_HARD, [0, 1]).all()),
    "soft_0_1": bool((Y_SOFT >= 0).all() and (Y_SOFT <= 1).all()),
    "soft_implies_hard": bool((Y_SOFT > 0).sum() == ((Y_SOFT > 0) & (Y_HARD == 1)).sum()),
    "official_folds_present": sorted(master["strat_fold"].dropna().unique().astype(int).tolist()),
    "diag_pos_weights": dict(zip(CLASSES, DIAG_POS_WEIGHTS.tolist())),
    "bin_weights": BIN_WEIGHTS,
}
save_json(sanity, MANIFEST_DIR / "sanity_checks.json")
log("Auditoria de datos terminada.")


In [ ]:
# ============================================================
# 8. Copia local reanudable de records100, estilo Ronald
# ============================================================
def count_record_files(path):
    if not path.exists():
        return 0
    return sum(1 for p in path.rglob("*") if p.suffix.lower() in [".hea", ".dat"])

def copy_records100_local(src: Path, dst: Path):
    if not USE_LOCAL_RECORDS100 or not IN_COLAB:
        return src
    marker = dst / ".copy_complete"
    src_count = count_record_files(src)
    dst_count = count_record_files(dst)
    if marker.exists() and dst_count >= src_count and src_count > 0:
        print("Usando records100 local ya preparado:", dst)
        return dst
    print("Copiando records100 desde Drive a /content para acelerar I/O.")
    print("Origen:", src)
    print("Destino:", dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        shutil.rmtree(dst)
    dst.mkdir(parents=True, exist_ok=True)
    try:
        subprocess.run(["rsync", "-a", f"{src}/", f"{dst}/"], check=True)
    except Exception as e:
        print("rsync no disponible o fallo; usando shutil.copytree:", repr(e))
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    marker.write_text(datetime.now().isoformat(), encoding="utf-8")
    return dst

ACTIVE_RECORDS100 = copy_records100_local(RECORDS100_PATH, LOCAL_RECORDS100)
print("ACTIVE_RECORDS100:", ACTIVE_RECORDS100)


In [ ]:
# ============================================================
# 9. Dataset WFDB y DataLoaders
# ============================================================
def resolve_record_stem(row):
    # PTB-XL usa filename_lr como records100/00000/00001_lr.
    if "filename_lr" in row and isinstance(row["filename_lr"], str) and row["filename_lr"]:
        rel = Path(row["filename_lr"])
        if "records100" in rel.parts:
            rel = Path(*rel.parts[rel.parts.index("records100") + 1:])
        return ACTIVE_RECORDS100 / rel
    ecg_id = int(row["ecg_id"])
    folder = f"{(ecg_id // 1000) * 1000:05d}"
    fname = f"{ecg_id:05d}_lr"
    return ACTIVE_RECORDS100 / folder / fname

def read_ecg_record(stem: Path):
    signal, fields = wfdb.rdsamp(str(stem))
    x = signal.astype(np.float32)
    if x.shape == (1000, 12):
        x = x.T
    elif x.shape == (12, 1000):
        pass
    else:
        raise ValueError(f"Shape ECG inesperado {x.shape} para {stem}")
    mean = x.mean(axis=1, keepdims=True)
    std = x.std(axis=1, keepdims=True)
    x = (x - mean) / (std + 1e-8)
    return x.astype(np.float32)

class PTBXLHCGDataset(Dataset):
    def __init__(self, indices):
        self.indices = np.asarray(indices)
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        idx = int(self.indices[i])
        row = master.iloc[idx]
        x = read_ecg_record(resolve_record_stem(row))
        bin_label = str(AMBIGUITY_BINS[idx])
        bin_idx = {"B0": 0, "B1": 1, "B2": 2, "B3": 3}[bin_label]
        item = {
            "x": torch.tensor(x, dtype=torch.float32),
            "y_hard": torch.tensor(Y_HARD[idx], dtype=torch.float32),
            "y_soft": torch.tensor(Y_SOFT[idx], dtype=torch.float32),
            "ambiguity": torch.tensor([AMBIGUITY[idx]], dtype=torch.float32),
            "review_binary": torch.tensor([REVIEW_BINARY[idx]], dtype=torch.float32),
            "bin_idx": torch.tensor(bin_idx, dtype=torch.long),
            "global_idx": torch.tensor(idx, dtype=torch.long),
        }
        return item

def make_loader(indices, shuffle=False):
    ds = PTBXLHCGDataset(indices)
    kwargs = dict(batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=False)
    if NUM_WORKERS > 0:
        kwargs.update(dict(persistent_workers=PERSISTENT_WORKERS, prefetch_factor=PREFETCH_FACTOR))
    return DataLoader(ds, **kwargs)

train_loader = make_loader(idx_train, shuffle=True)
val_loader = make_loader(idx_val, shuffle=False)
test_loader = make_loader(idx_test, shuffle=False)

# Sanity check de WFDB.
batch = next(iter(train_loader))
print("batch x:", batch["x"].shape, "y:", batch["y_hard"].shape)
assert batch["x"].shape[1:] == (12, 1000)


In [ ]:
# ============================================================
# 10. Arquitectura CNN1D multi-task
# ============================================================
class ConvBlock(nn.Module):
    def __init__(self, c_in, c_out, kernel, dropout=0.0, pool=True):
        super().__init__()
        pad = kernel // 2
        layers = [
            nn.Conv1d(c_in, c_out, kernel_size=kernel, padding=pad),
            nn.BatchNorm1d(c_out),
            nn.ReLU(inplace=True),
        ]
        if pool:
            layers.append(nn.MaxPool1d(2))
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

class CNN1DBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            ConvBlock(12, 32, 7, dropout=0.20, pool=True),
            ConvBlock(32, 64, 7, dropout=0.25, pool=True),
            ConvBlock(64, 128, 5, dropout=0.25, pool=True),
            ConvBlock(128, 256, 5, dropout=0.0, pool=False),
            nn.AdaptiveAvgPool1d(1),
        )
    def forward(self, x):
        z = self.net(x).squeeze(-1)
        return z

class HCGSelectiveECGModel(nn.Module):
    def __init__(self, multitask=False, n_classes=5):
        super().__init__()
        self.multitask = multitask
        self.backbone = CNN1DBackbone()
        self.dropout = nn.Dropout(0.30)
        self.diag_head = nn.Linear(256, n_classes)
        self.review_head = nn.Linear(256, 1) if multitask else None
    def forward(self, x):
        z = self.dropout(self.backbone(x))
        diag_logits = self.diag_head(z)
        if self.multitask:
            review_logit = self.review_head(z)
            return {"diag_logits": diag_logits, "review_logit": review_logit}
        return {"diag_logits": diag_logits}

def make_model(model_name):
    return HCGSelectiveECGModel(multitask=model_name.startswith("P")).to(DEVICE)


In [ ]:
# ============================================================
# 11. Metricas diagnosticas, calibracion y revision
# ============================================================
def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-x))

def safe_auc(y_true, y_score):
    try:
        if len(np.unique(y_true)) < 2:
            return np.nan
        return roc_auc_score(y_true, y_score)
    except Exception:
        return np.nan

def safe_auprc(y_true, y_score):
    try:
        if len(np.unique(y_true)) < 2:
            return np.nan
        return average_precision_score(y_true, y_score)
    except Exception:
        return np.nan

def ece_binary(y_true, p, n_bins=10):
    y_true = np.asarray(y_true).astype(float)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (p >= lo) & (p < hi if hi < 1.0 else p <= hi)
        if mask.sum() == 0:
            continue
        conf = p[mask].mean()
        acc = y_true[mask].mean()
        ece += (mask.sum() / len(p)) * abs(acc - conf)
    return float(ece)

def diagnostic_metrics(y_true, probs, thresholds=None, prefix=""):
    y_true = np.asarray(y_true).astype(int)
    probs = np.asarray(probs).astype(float)
    if thresholds is None:
        thresholds = np.full(y_true.shape[1], 0.5)
    pred = (probs >= thresholds.reshape(1, -1)).astype(int)
    rows_class = []
    for k, cls in enumerate(CLASSES):
        rows_class.append({
            "class": cls,
            "AUROC": safe_auc(y_true[:, k], probs[:, k]),
            "AUPRC": safe_auprc(y_true[:, k], probs[:, k]),
            "F1": f1_score(y_true[:, k], pred[:, k], zero_division=0),
            "Precision": precision_score(y_true[:, k], pred[:, k], zero_division=0),
            "Recall": recall_score(y_true[:, k], pred[:, k], zero_division=0),
            "Brier": brier_score_loss(y_true[:, k], probs[:, k]) if len(np.unique(y_true[:, k])) > 1 else np.nan,
            "ECE": ece_binary(y_true[:, k], probs[:, k]),
        })
    by_class = pd.DataFrame(rows_class)
    global_metrics = {
        prefix + "AUROC_macro": float(np.nanmean(by_class["AUROC"])),
        prefix + "AUPRC_macro": float(np.nanmean(by_class["AUPRC"])),
        prefix + "F1_macro": f1_score(y_true, pred, average="macro", zero_division=0),
        prefix + "Precision_macro": precision_score(y_true, pred, average="macro", zero_division=0),
        prefix + "Recall_macro": recall_score(y_true, pred, average="macro", zero_division=0),
        prefix + "Brier_macro": float(np.nanmean(by_class["Brier"])),
        prefix + "ECE_macro": float(np.nanmean(by_class["ECE"])),
    }
    try:
        global_metrics[prefix + "AUROC_micro"] = roc_auc_score(y_true.ravel(), probs.ravel())
        global_metrics[prefix + "AUPRC_micro"] = average_precision_score(y_true.ravel(), probs.ravel())
    except Exception:
        global_metrics[prefix + "AUROC_micro"] = np.nan
        global_metrics[prefix + "AUPRC_micro"] = np.nan
    return global_metrics, by_class, pred

def select_thresholds_val(y_true, probs):
    thresholds = []
    grid = np.linspace(0.05, 0.95, 91)
    for k in range(y_true.shape[1]):
        best_t, best_f = 0.5, -1
        for t in grid:
            f = f1_score(y_true[:, k], probs[:, k] >= t, zero_division=0)
            if f > best_f:
                best_f, best_t = f, t
        thresholds.append(best_t)
    return np.array(thresholds, dtype=np.float32)

def review_metrics(y_review, ambiguity, score, threshold=None):
    y_review = np.asarray(y_review).astype(int).ravel()
    ambiguity = np.asarray(ambiguity).astype(float).ravel()
    score = np.asarray(score).astype(float).ravel()
    if threshold is None:
        grid = np.linspace(0.01, 0.99, 99)
        fs = [f1_score(y_review, score >= t, zero_division=0) for t in grid]
        threshold = float(grid[int(np.argmax(fs))])
    pred = (score >= threshold).astype(int)
    sp = spearmanr(ambiguity, score, nan_policy="omit").correlation
    return {
        "review_AUROC": safe_auc(y_review, score),
        "review_AUPRC": safe_auprc(y_review, score),
        "review_threshold": threshold,
        "review_accuracy": accuracy_score(y_review, pred),
        "review_precision": precision_score(y_review, pred, zero_division=0),
        "review_recall": recall_score(y_review, pred, zero_division=0),
        "review_F1": f1_score(y_review, pred, zero_division=0),
        "MAE_ambiguity": mean_absolute_error(ambiguity, score),
        "RMSE_ambiguity": math.sqrt(mean_squared_error(ambiguity, score)),
        "Spearman_ambiguity": float(sp) if sp == sp else np.nan,
    }


In [ ]:
# ============================================================
# 12. Entrenamiento reanudable
# ============================================================
POS_WEIGHT_TENSOR = torch.tensor(DIAG_POS_WEIGHTS, dtype=torch.float32, device=DEVICE)
BIN_WEIGHT_TENSOR = torch.tensor(
    [BIN_WEIGHTS["B0"], BIN_WEIGHTS["B1"], BIN_WEIGHTS["B2"], BIN_WEIGHTS["B3"]],
    dtype=torch.float32, device=DEVICE
)

def compute_loss(outputs, batch, model_name, epoch):
    diag_logits = outputs["diag_logits"]
    if model_name == "B2_SOFT_DIRECT":
        diag_target = batch["y_soft"].to(DEVICE)
        diag_loss = F.binary_cross_entropy_with_logits(diag_logits, diag_target)
    else:
        diag_target = batch["y_hard"].to(DEVICE)
        diag_loss = F.binary_cross_entropy_with_logits(diag_logits, diag_target, pos_weight=POS_WEIGHT_TENSOR)

    review_loss_value = torch.tensor(0.0, device=DEVICE)
    if model_name.startswith("P") and epoch >= WARMUP_EPOCHS_MTL:
        pred = torch.sigmoid(outputs["review_logit"])
        target = batch["ambiguity"].to(DEVICE)
        per_sample = F.smooth_l1_loss(pred, target, reduction="none").squeeze(1)
        if model_name == "P1_MTL_BALANCED_PROPOSED":
            weights = BIN_WEIGHT_TENSOR[batch["bin_idx"].to(DEVICE)]
            review_loss_value = (per_sample * weights).sum() / weights.sum().clamp_min(1e-8)
        else:
            review_loss_value = per_sample.mean()
    total = diag_loss + REVIEW_LOSS_WEIGHT * review_loss_value
    return total, float(diag_loss.detach().cpu()), float(review_loss_value.detach().cpu())

@torch.no_grad()
def run_inference(model, loader, model_name):
    model.eval()
    logits_all, probs_all, review_all, gidx_all = [], [], [], []
    yhard_all, ysoft_all, amb_all, revbin_all, binidx_all = [], [], [], [], []
    for batch in tqdm(loader, desc=f"Inferencia {model_name}", leave=False):
        x = batch["x"].to(DEVICE)
        out = model(x)
        logits = out["diag_logits"].detach().cpu().numpy()
        probs = sigmoid_np(logits)
        if "review_logit" in out:
            review = torch.sigmoid(out["review_logit"]).detach().cpu().numpy().reshape(-1)
        else:
            review = np.full(len(logits), np.nan)
        logits_all.append(logits); probs_all.append(probs); review_all.append(review)
        yhard_all.append(batch["y_hard"].numpy()); ysoft_all.append(batch["y_soft"].numpy())
        amb_all.append(batch["ambiguity"].numpy().reshape(-1))
        revbin_all.append(batch["review_binary"].numpy().reshape(-1))
        binidx_all.append(batch["bin_idx"].numpy())
        gidx_all.append(batch["global_idx"].numpy())
    return {
        "logits": np.vstack(logits_all),
        "probs": np.vstack(probs_all),
        "review_score": np.concatenate(review_all),
        "y_hard": np.vstack(yhard_all),
        "y_soft": np.vstack(ysoft_all),
        "ambiguity": np.concatenate(amb_all),
        "review_binary": np.concatenate(revbin_all),
        "bin_idx": np.concatenate(binidx_all),
        "global_idx": np.concatenate(gidx_all),
    }

def val_summary(model, loader, model_name):
    pred = run_inference(model, loader, model_name)
    gm, _, _ = diagnostic_metrics(pred["y_hard"], pred["probs"])
    out = {
        "val_diag_auroc_macro": gm["AUROC_macro"],
        "val_diag_auprc_macro": gm["AUPRC_macro"],
        "val_diag_f1_macro_05": gm["F1_macro"],
    }
    if model_name.startswith("P"):
        rm = review_metrics(pred["review_binary"], pred["ambiguity"], pred["review_score"])
        out["val_review_auprc"] = rm["review_AUPRC"]
        out["val_review_auroc"] = rm["review_AUROC"]
    return out

def save_checkpoint(path, model, optimizer, scheduler, epoch, best_score, model_name, config_extra=None):
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler else None,
        "epoch": epoch,
        "best_score": best_score,
        "model_name": model_name,
        "config": {**main_config, **(config_extra or {})},
        "timestamp": datetime.now().isoformat(),
    }, path)

def load_checkpoint(path, model, optimizer=None, scheduler=None):
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    if optimizer is not None and ckpt.get("optimizer_state_dict"):
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if scheduler is not None and ckpt.get("scheduler_state_dict"):
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    return ckpt

BASELINE_B1_VAL_AUROC = None

def train_model(model_name):
    global BASELINE_B1_VAL_AUROC
    model_dir = MODELS_DIR / model_name
    ckpt_dir = model_dir / "checkpoints"
    model_dir.mkdir(parents=True, exist_ok=True)
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    history_path = model_dir / "history.csv"
    best_path = ckpt_dir / "best.pt"
    last_path = ckpt_dir / "last.pt"
    val_pred_path = model_dir / "val_predictions.csv"
    test_pred_path = model_dir / "test_predictions.csv"

    if SKIP_COMPLETED and not FORCE_RETRAIN and best_path.exists() and history_path.exists() and test_pred_path.exists():
        log(f"{model_name}: ya completado, se salta.")
        model = make_model(model_name)
        load_checkpoint(best_path, model)
        if model_name == "B1_HARD_DIAG":
            hist = pd.read_csv(history_path)
            BASELINE_B1_VAL_AUROC = float(hist["val_diag_auroc_macro"].max())
        return model

    model = make_model(model_name)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
    scaler = torch.cuda.amp.GradScaler(enabled=MIXED_PRECISION)
    start_epoch = 0
    best_score = -np.inf
    best_fallback = -np.inf
    no_improve = 0
    history = []

    if RESUME and last_path.exists() and not FORCE_RETRAIN:
        ckpt = load_checkpoint(last_path, model, optimizer, scheduler)
        start_epoch = int(ckpt.get("epoch", -1)) + 1
        best_score = float(ckpt.get("best_score", -np.inf))
        if history_path.exists():
            history = pd.read_csv(history_path).to_dict("records")
        log(f"{model_name}: reanudando desde epoch {start_epoch}.")

    for epoch in range(start_epoch, MAX_EPOCHS):
        t0 = time.time()
        model.train()
        losses, diag_losses, rev_losses = [], [], []
        for batch in tqdm(train_loader, desc=f"{model_name} epoch {epoch+1}/{MAX_EPOCHS}", leave=False):
            optimizer.zero_grad(set_to_none=True)
            x = batch["x"].to(DEVICE)
            with torch.cuda.amp.autocast(enabled=MIXED_PRECISION):
                out = model(x)
                loss, diag_loss, review_loss = compute_loss(out, batch, model_name, epoch)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
            losses.append(float(loss.detach().cpu()))
            diag_losses.append(diag_loss)
            rev_losses.append(review_loss)

        metrics = val_summary(model, val_loader, model_name)
        score_diag = metrics["val_diag_auprc_macro"]
        if model_name.startswith("P"):
            b1_ref = BASELINE_B1_VAL_AUROC if BASELINE_B1_VAL_AUROC is not None else -np.inf
            valid_diag = metrics["val_diag_auroc_macro"] >= (b1_ref - 0.01)
            if valid_diag:
                selection_score = metrics.get("val_review_auprc", -np.inf)
                selection_reason = "review_auprc_with_diag_constraint"
            else:
                selection_score = metrics["val_diag_auroc_macro"] - 10.0
                selection_reason = "diag_constraint_not_met"
            best_fallback = max(best_fallback, metrics["val_diag_auroc_macro"])
        else:
            selection_score = score_diag
            selection_reason = "diag_auprc_macro"

        scheduler.step(score_diag if score_diag == score_diag else 0.0)
        improved = selection_score > best_score
        if improved:
            best_score = float(selection_score)
            no_improve = 0
            save_checkpoint(best_path, model, optimizer, scheduler, epoch, best_score, model_name)
        else:
            no_improve += 1
        save_checkpoint(last_path, model, optimizer, scheduler, epoch, best_score, model_name)

        row = {
            "epoch": epoch,
            "train_loss": float(np.mean(losses)),
            "train_diag_loss": float(np.mean(diag_losses)),
            "train_review_loss": float(np.mean(rev_losses)),
            "lr": optimizer.param_groups[0]["lr"],
            "elapsed_sec": time.time() - t0,
            "checkpoint_saved": bool(improved),
            "selection_score": best_score,
            "selection_reason": selection_reason,
            **metrics,
        }
        history.append(row)
        pd.DataFrame(history).to_csv(history_path, index=False)
        log(f"{model_name} epoch {epoch+1}: {row}")
        if no_improve >= EARLY_STOPPING_PATIENCE:
            log(f"{model_name}: early stopping.")
            break

    load_checkpoint(best_path, model)
    pred_val = run_inference(model, val_loader, model_name)
    pred_test = run_inference(model, test_loader, model_name)
    save_predictions(model_name, "val", pred_val, val_pred_path)
    save_predictions(model_name, "test", pred_test, test_pred_path)

    hist = pd.read_csv(history_path)
    selection_summary = {
        "model_name": model_name,
        "best_score": float(best_score),
        "best_epoch": int(hist.loc[hist["selection_score"].idxmax(), "epoch"]) if "selection_score" in hist else None,
        "baseline_b1_val_auroc": BASELINE_B1_VAL_AUROC,
        "selection_policy": "P models require diagnostic AUROC within 0.01 of B1, then maximize review AUPRC.",
    }
    save_json(selection_summary, model_dir / "selection_summary.json")
    save_json({"model_name": model_name, **main_config}, model_dir / "config.json")

    if model_name == "B1_HARD_DIAG":
        BASELINE_B1_VAL_AUROC = float(hist["val_diag_auroc_macro"].max())
    return model

def save_predictions(model_name, split, pred, path):
    meta = master.iloc[pred["global_idx"]].reset_index(drop=True).copy()
    out = pd.DataFrame({
        "ecg_id": meta["ecg_id"].values,
        "global_idx": pred["global_idx"],
        "strat_fold": meta["strat_fold"].values,
        "age": meta["age"].values if "age" in meta else np.nan,
        "sex": meta["sex"].values if "sex" in meta else np.nan,
        "patient_id": meta["patient_id"].values if "patient_id" in meta else np.nan,
        "filename_lr": meta["filename_lr"].values if "filename_lr" in meta else "",
        "ambiguity_score": pred["ambiguity"],
        "review_binary": pred["review_binary"].astype(int),
    })
    bin_names = np.array(["B0", "B1", "B2", "B3"])
    out["ambiguity_bin"] = bin_names[pred["bin_idx"]]
    for k, cls in enumerate(CLASSES):
        out[f"y_{cls}"] = pred["y_hard"][:, k].astype(int)
        out[f"soft_{cls}"] = pred["y_soft"][:, k]
        out[f"prob_{cls}"] = pred["probs"][:, k]
        out[f"logit_{cls}"] = pred["logits"][:, k]
    if not np.isnan(pred["review_score"]).all():
        out["review_score"] = pred["review_score"]
    out.to_csv(path, index=False)
    return out


In [ ]:
# ============================================================
# 13. Entrenar modelos
# ============================================================
MODEL_ORDER = [
    "B1_HARD_DIAG",
    "B2_SOFT_DIRECT",
    "P0_MTL_UNBALANCED",
    "P1_MTL_BALANCED_PROPOSED",
]

trained_models = {}
for model_name in MODEL_ORDER:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    trained_models[model_name] = train_model(model_name)

log("Entrenamiento/inferencia base completados.")


In [ ]:
# ============================================================
# 14. Cargar predicciones, umbrales y baseline B3
# ============================================================
def load_model_predictions(model_name, split):
    return pd.read_csv(MODELS_DIR / model_name / f"{split}_predictions.csv")

preds = {m: {s: load_model_predictions(m, s) for s in ["val", "test"]} for m in MODEL_ORDER}

thresholds = {}
for m in MODEL_ORDER:
    val_df = preds[m]["val"]
    yv = val_df[[f"y_{c}" for c in CLASSES]].values
    pv = val_df[[f"prob_{c}" for c in CLASSES]].values
    thresholds[m] = select_thresholds_val(yv, pv)

save_json({m: dict(zip(CLASSES, thresholds[m].tolist())) for m in MODEL_ORDER}, CONFIG_DIR / "validated_thresholds.json")

def threshold_uncertainty_score(probs, tau):
    tau = tau.reshape(1, -1)
    denom = np.maximum(tau, 1 - tau)
    u = 1 - np.abs(probs - tau) / denom
    return np.clip(u, 0, 1).max(axis=1)

for split in ["val", "test"]:
    b1 = preds["B1_HARD_DIAG"][split]
    probs = b1[[f"prob_{c}" for c in CLASSES]].values
    score = threshold_uncertainty_score(probs, thresholds["B1_HARD_DIAG"])
    b3 = b1.copy()
    b3["review_score"] = score
    preds.setdefault("B3_THRESHOLD_UNCERTAINTY", {})[split] = b3

    b2 = preds["B2_SOFT_DIRECT"][split].copy()
    b2_probs = b2[[f"prob_{c}" for c in CLASSES]].values
    b2["review_score"] = np.max(1 - np.abs(2 * b2_probs - 1), axis=1)
    preds.setdefault("B2_PROB_UNCERTAINTY", {})[split] = b2

log("Umbrales validados y scores B3/B2 creados.")


In [ ]:
# ============================================================
# 15. Metricas diagnosticas y de revision
# ============================================================
diag_global_rows = []
diag_class_rows = []
review_rows = []

for m in MODEL_ORDER:
    for split in ["val", "test"]:
        df = preds[m][split]
        y = df[[f"y_{c}" for c in CLASSES]].values
        p = df[[f"prob_{c}" for c in CLASSES]].values
        gm05, bc05, _ = diagnostic_metrics(y, p, thresholds=np.full(len(CLASSES), 0.5))
        gmthr, bcthr, pred_thr = diagnostic_metrics(y, p, thresholds=thresholds[m])
        row = {"model": m, "split": split}
        row.update({k + "_05": v for k, v in gm05.items()})
        row.update({k + "_thrval": v for k, v in gmthr.items()})
        diag_global_rows.append(row)
        bc05["model"] = m; bc05["split"] = split; bc05["threshold_type"] = "0.5"
        bcthr["model"] = m; bcthr["split"] = split; bcthr["threshold_type"] = "thrval"
        diag_class_rows.extend(bc05.to_dict("records"))
        diag_class_rows.extend(bcthr.to_dict("records"))

        # Agregar predicciones por umbral validado a tablas maestras.
        for k, cls in enumerate(CLASSES):
            df[f"pred_{cls}_thrval"] = pred_thr[:, k]
        df["sample_error"] = (pred_thr != y).any(axis=1).astype(int)
        preds[m][split] = df

for score_name in ["B2_PROB_UNCERTAINTY", "B3_THRESHOLD_UNCERTAINTY", "P0_MTL_UNBALANCED", "P1_MTL_BALANCED_PROPOSED"]:
    val_df = preds[score_name]["val"]
    test_df = preds[score_name]["test"]
    val_metrics = review_metrics(val_df["review_binary"], val_df["ambiguity_score"], val_df["review_score"])
    test_metrics = review_metrics(
        test_df["review_binary"], test_df["ambiguity_score"], test_df["review_score"],
        threshold=val_metrics["review_threshold"]
    )
    row = {"model_or_score": score_name, "split": "test", **test_metrics}
    review_rows.append(row)

    # MAE por bin.
    for b in ["B0", "B1", "B2", "B3"]:
        mask = test_df["ambiguity_bin"] == b
        row[f"MAE_{b}"] = mean_absolute_error(
            test_df.loc[mask, "ambiguity_score"], test_df.loc[mask, "review_score"]
        ) if mask.sum() else np.nan

diag_global = pd.DataFrame(diag_global_rows)
diag_by_class = pd.DataFrame(diag_class_rows)
review_table = pd.DataFrame(review_rows)

diag_global.to_csv(TABLES_DIR / "diagnostic_metrics_global.csv", index=False)
diag_by_class.to_csv(TABLES_DIR / "diagnostic_metrics_by_class.csv", index=False)
review_table.to_csv(TABLES_DIR / "review_metrics.csv", index=False)
diag_global.to_csv(FIGURE_DATA_DIR / "fig03_diagnostic_metrics.csv", index=False)

display(diag_global[diag_global["split"] == "test"])
display(review_table)


In [ ]:
# ============================================================
# 16. Derivacion selectiva
# ============================================================
def accepted_metrics_for_score(score_df, diag_model_name, q):
    df = score_df.copy()
    diag_df = preds[diag_model_name]["test"].set_index("ecg_id")
    df = df.set_index("ecg_id")
    common = df.index.intersection(diag_df.index)
    df = df.loc[common].copy()
    diag = diag_df.loc[common].copy()
    n = len(df)
    n_ref = max(1, int(round(q * n)))
    order = np.argsort(-df["review_score"].values)
    referred = np.zeros(n, dtype=bool)
    referred[order[:n_ref]] = True
    accepted = ~referred

    y = diag[[f"y_{c}" for c in CLASSES]].values
    p = diag[[f"prob_{c}" for c in CLASSES]].values
    tau = thresholds[diag_model_name]
    gm, _, pred = diagnostic_metrics(y[accepted], p[accepted], thresholds=tau)
    sample_error = (pred != y[accepted]).any(axis=1).astype(int)

    full_pred = (p >= tau.reshape(1, -1)).astype(int)
    full_error = (full_pred != y).any(axis=1)
    total_errors = max(1, int(full_error.sum()))
    total_lowcert = max(1, int(df["review_binary"].sum()))

    row = {
        "model_or_score": df.attrs.get("score_name", "unknown"),
        "diag_model": diag_model_name,
        "q": q,
        "coverage": float(accepted.mean()),
        "n_referred": int(referred.sum()),
        "n_accepted": int(accepted.sum()),
        "ErrorCapture@q": float((referred & full_error).sum() / total_errors),
        "LowCertCapture@q": float((referred & (df["review_binary"].values == 1)).sum() / total_lowcert),
        "F1_accepted": gm["F1_macro"],
        "Brier_accepted": gm["Brier_macro"],
        "ECE_accepted": gm["ECE_macro"],
    }
    return row, referred, full_pred, y

selective_rows = []
by_class_rows = []
score_to_diag = {
    "B3_THRESHOLD_UNCERTAINTY": "B1_HARD_DIAG",
    "P0_MTL_UNBALANCED": "P0_MTL_UNBALANCED",
    "P1_MTL_BALANCED_PROPOSED": "P1_MTL_BALANCED_PROPOSED",
}

rng = np.random.default_rng(SEED)
random_df = preds["B1_HARD_DIAG"]["test"].copy()
random_df["review_score"] = rng.random(len(random_df))
preds["RANDOM_REFERRAL"] = {"test": random_df}
score_to_diag["RANDOM_REFERRAL"] = "B1_HARD_DIAG"

for score_name, diag_name in score_to_diag.items():
    score_df = preds[score_name]["test"].copy()
    score_df.attrs["score_name"] = score_name
    for q in Q_VALUES:
        row, referred, full_pred, y = accepted_metrics_for_score(score_df, diag_name, q)
        row["model_or_score"] = score_name
        selective_rows.append(row)
        for k, cls in enumerate(CLASSES):
            fn = (y[:, k] == 1) & (full_pred[:, k] == 0)
            fp = (y[:, k] == 0) & (full_pred[:, k] == 1)
            ce = full_pred[:, k] != y[:, k]
            by_class_rows.append({
                "model_or_score": score_name,
                "diag_model": diag_name,
                "q": q,
                "class": cls,
                "ClassErrorCapture@q": float((referred & ce).sum() / max(1, ce.sum())),
                "FalseNegativeCapture@q": float((referred & fn).sum() / max(1, fn.sum())),
                "FalsePositiveCapture@q": float((referred & fp).sum() / max(1, fp.sum())),
                "n_false_negative": int(fn.sum()),
                "n_class_error": int(ce.sum()),
            })

selective_table = pd.DataFrame(selective_rows)
selective_by_class = pd.DataFrame(by_class_rows)
selective_table.to_csv(TABLES_DIR / "selective_referral_metrics.csv", index=False)
selective_by_class.to_csv(TABLES_DIR / "selective_referral_by_class.csv", index=False)
selective_table.to_csv(FIGURE_DATA_DIR / "fig05_error_capture.csv", index=False)

balance_rows = []
for m in ["P0_MTL_UNBALANCED", "P1_MTL_BALANCED_PROPOSED"]:
    rev = review_table[review_table["model_or_score"] == m].iloc[0].to_dict()
    sel20 = selective_table[(selective_table["model_or_score"] == m) & (np.isclose(selective_table["q"], 0.20))].iloc[0].to_dict()
    balance_rows.append({
        "model": m,
        "review_AUPRC": rev.get("review_AUPRC", np.nan),
        "MAE_B0": rev.get("MAE_B0", np.nan),
        "MAE_B1": rev.get("MAE_B1", np.nan),
        "MAE_B2": rev.get("MAE_B2", np.nan),
        "MAE_B3": rev.get("MAE_B3", np.nan),
        "ErrorCapture@20": sel20.get("ErrorCapture@q", np.nan),
        "LowCertCapture@20": sel20.get("LowCertCapture@q", np.nan),
    })
balance_effect = pd.DataFrame(balance_rows)
balance_effect.to_csv(TABLES_DIR / "balance_effect.csv", index=False)

display(selective_table)
display(selective_by_class[(selective_by_class["q"] == 0.20) & (selective_by_class["class"].isin(["MI", "HYP"]))])
display(balance_effect)


In [ ]:
# ============================================================
# 17. Master predictions val/test
# ============================================================
def build_master_predictions(split):
    base = preds["B1_HARD_DIAG"][split][[
        "ecg_id", "global_idx", "strat_fold", "age", "sex", "patient_id", "filename_lr",
        "ambiguity_score", "ambiguity_bin", "review_binary"
    ]].copy()
    for cls in CLASSES:
        base[f"y_{cls}"] = preds["B1_HARD_DIAG"][split][f"y_{cls}"]
        base[f"soft_{cls}"] = preds["B1_HARD_DIAG"][split][f"soft_{cls}"]
        base[f"likelihood_{cls}"] = preds["B1_HARD_DIAG"][split][f"soft_{cls}"]

    for m in MODEL_ORDER:
        df = preds[m][split].set_index("ecg_id")
        for cls in CLASSES:
            base[f"{m}_prob_{cls}"] = base["ecg_id"].map(df[f"prob_{cls}"])
            base[f"{m}_pred_{cls}_thrval"] = base["ecg_id"].map(df[f"pred_{cls}_thrval"])
        base[f"{m}_sample_error"] = base["ecg_id"].map(df["sample_error"])
        if "review_score" in df.columns:
            base[f"{m}_review_score"] = base["ecg_id"].map(df["review_score"])

    b3 = preds["B3_THRESHOLD_UNCERTAINTY"][split].set_index("ecg_id")
    base["B3_threshold_uncertainty_score"] = base["ecg_id"].map(b3["review_score"])

    if split == "test":
        for score_name in ["B3_THRESHOLD_UNCERTAINTY", "P0_MTL_UNBALANCED", "P1_MTL_BALANCED_PROPOSED", "RANDOM_REFERRAL"]:
            score_df = preds[score_name]["test"].set_index("ecg_id")
            score = base["ecg_id"].map(score_df["review_score"]).values
            for q in Q_VALUES:
                n_ref = max(1, int(round(q * len(score))))
                order = np.argsort(-score)
                ref = np.zeros(len(score), dtype=int)
                ref[order[:n_ref]] = 1
                base[f"{score_name}_referred_top{int(q*100):02d}"] = ref
    return base

master_val = build_master_predictions("val")
master_test = build_master_predictions("test")
master_val.to_csv(TABLES_DIR / "master_predictions_val.csv", index=False)
master_test.to_csv(TABLES_DIR / "master_predictions_test.csv", index=False)
print(TABLES_DIR / "master_predictions_test.csv")


In [ ]:
# ============================================================
# 18. Bootstrap pareado
# ============================================================
def metric_auprc_macro_from_df(df, model_name):
    y = df[[f"y_{c}" for c in CLASSES]].values
    p = df[[f"{model_name}_prob_{c}" for c in CLASSES]].values
    return np.nanmean([safe_auprc(y[:, k], p[:, k]) for k in range(len(CLASSES))])

def review_auprc_from_master(df, score_col):
    return safe_auprc(df["review_binary"].values, df[score_col].values)

def error_capture_20_from_master(df, score_col, diag_model):
    y = df[[f"y_{c}" for c in CLASSES]].values
    pred = df[[f"{diag_model}_pred_{c}_thrval" for c in CLASSES]].values
    err = (pred != y).any(axis=1)
    score = df[score_col].values
    n_ref = max(1, int(round(0.20 * len(df))))
    referred = np.zeros(len(df), dtype=bool)
    referred[np.argsort(-score)[:n_ref]] = True
    return float((referred & err).sum() / max(1, err.sum()))

def bootstrap_summary(master_df, n_boot=N_BOOTSTRAP):
    rng = np.random.default_rng(SEED)
    if "patient_id" in master_df.columns and master_df["patient_id"].notna().any():
        units = master_df["patient_id"].fillna(master_df["ecg_id"]).values
        unique_units = np.unique(units)
        sample_indices_by_unit = {u: np.where(units == u)[0] for u in unique_units}
        def sample_df():
            sampled_units = rng.choice(unique_units, size=len(unique_units), replace=True)
            idx = np.concatenate([sample_indices_by_unit[u] for u in sampled_units])
            return master_df.iloc[idx].reset_index(drop=True)
        method = "patient"
    else:
        def sample_df():
            idx = rng.choice(np.arange(len(master_df)), size=len(master_df), replace=True)
            return master_df.iloc[idx].reset_index(drop=True)
        method = "ecg"

    stats = []
    for _ in tqdm(range(n_boot), desc="Bootstrap"):
        b = sample_df()
        stats.append({
            "diff_P1_minus_B1_AUPRC_macro": metric_auprc_macro_from_df(b, "P1_MTL_BALANCED_PROPOSED") - metric_auprc_macro_from_df(b, "B1_HARD_DIAG"),
            "diff_P1_minus_B3_review_AUPRC": review_auprc_from_master(b, "P1_MTL_BALANCED_PROPOSED_review_score") - review_auprc_from_master(b, "B3_threshold_uncertainty_score"),
            "diff_P1_minus_B3_ErrorCapture20": error_capture_20_from_master(b, "P1_MTL_BALANCED_PROPOSED_review_score", "P1_MTL_BALANCED_PROPOSED") - error_capture_20_from_master(b, "B3_threshold_uncertainty_score", "B1_HARD_DIAG"),
            "diff_P1_minus_P0_review_AUPRC": review_auprc_from_master(b, "P1_MTL_BALANCED_PROPOSED_review_score") - review_auprc_from_master(b, "P0_MTL_UNBALANCED_review_score"),
        })
    boot = pd.DataFrame(stats)
    rows = []
    for col in boot.columns:
        rows.append({
            "comparison": col,
            "mean": boot[col].mean(),
            "ci95_low": boot[col].quantile(0.025),
            "ci95_high": boot[col].quantile(0.975),
            "bootstrap_unit": method,
            "n_bootstrap": n_boot,
        })
    return pd.DataFrame(rows), boot

bootstrap_table, bootstrap_samples = bootstrap_summary(master_test)
bootstrap_table.to_csv(TABLES_DIR / "bootstrap_summary.csv", index=False)
bootstrap_samples.to_csv(TABLES_DIR / "bootstrap_samples.csv", index=False)
display(bootstrap_table)


In [ ]:
# ============================================================
# 19. Subgrupos
# ============================================================
def subgroup_metrics():
    rows = []
    groups = []
    if "sex" in master_test.columns:
        for val in sorted(master_test["sex"].dropna().unique()):
            groups.append((f"sex_{val}", master_test["sex"] == val))
    if "age" in master_test.columns:
        groups.append(("age_lt_60", master_test["age"] < 60))
        groups.append(("age_ge_60", master_test["age"] >= 60))

    for group_name, mask in groups:
        sub = master_test[mask].copy()
        if len(sub) < 10:
            continue
        for model in ["B1_HARD_DIAG", "P1_MTL_BALANCED_PROPOSED"]:
            y = sub[[f"y_{c}" for c in CLASSES]].values
            p = sub[[f"{model}_prob_{c}" for c in CLASSES]].values
            pred = sub[[f"{model}_pred_{c}_thrval" for c in CLASSES]].values
            gm, _, _ = diagnostic_metrics(y, p, thresholds=np.array([0.5] * len(CLASSES)))
            score_col = f"{model}_review_score" if f"{model}_review_score" in sub else "B3_threshold_uncertainty_score"
            ec20 = error_capture_20_from_master(sub, score_col, model)
            rows.append({
                "model": model,
                "subgroup": group_name,
                "N": len(sub),
                "AUROC_macro": gm["AUROC_macro"],
                "AUPRC_macro": gm["AUPRC_macro"],
                "F1_macro": f1_score(y.ravel(), pred.ravel(), zero_division=0),
                "ECE_macro": gm["ECE_macro"],
                "review_AUPRC": safe_auprc(sub["review_binary"], sub[score_col]),
                "ErrorCapture@20": ec20,
                "referral_rate_top20": 0.20,
            })
    return pd.DataFrame(rows)

subgroup_table = subgroup_metrics()
subgroup_table.to_csv(TABLES_DIR / "subgroup_metrics.csv", index=False)
display(subgroup_table)


In [ ]:
# ============================================================
# 20. Explicabilidad: grad x input, lead ablation y perturbaciones
# ============================================================
LEADS = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]

def load_best_model(model_name):
    model = make_model(model_name)
    load_checkpoint(MODELS_DIR / model_name / "checkpoints" / "best.pt", model)
    model.eval()
    return model

def select_explainability_cases():
    df = master_test.copy()
    p1 = preds["P1_MTL_BALANCED_PROPOSED"]["test"].set_index("ecg_id")
    df["p1_review_score"] = df["ecg_id"].map(p1["review_score"])
    selections = []
    def add_case(case_type, sub):
        if len(sub) == 0:
            return
        row = sub.iloc[0]
        selections.append({"case_type": case_type, "ecg_id": row["ecg_id"], "global_idx": int(row["global_idx"])})
    add_case("clear_correct_accepted", df[(df["review_binary"] == 0) & (df["P1_MTL_BALANCED_PROPOSED_sample_error"] == 0)].sort_values("p1_review_score"))
    add_case("low_cert_referred", df[df["review_binary"] == 1].sort_values("p1_review_score", ascending=False))
    add_case("error_captured", df[df["P1_MTL_BALANCED_PROPOSED_sample_error"] == 1].sort_values("p1_review_score", ascending=False))
    add_case("error_not_captured", df[df["P1_MTL_BALANCED_PROPOSED_sample_error"] == 1].sort_values("p1_review_score"))
    add_case("MI_referred", df[df["y_MI"] == 1].sort_values("p1_review_score", ascending=False))
    add_case("HYP_referred", df[df["y_HYP"] == 1].sort_values("p1_review_score", ascending=False))
    out = pd.DataFrame(selections).drop_duplicates("ecg_id")
    out.to_csv(EXPLAIN_DIR / "case_selection.csv", index=False)
    return out

def grad_x_input(model, x, target="review", class_idx=None):
    model.zero_grad(set_to_none=True)
    x = x.clone().detach().to(DEVICE).unsqueeze(0)
    x.requires_grad_(True)
    out = model(x)
    if target == "review":
        score = torch.sigmoid(out["review_logit"])[0, 0]
    else:
        score = out["diag_logits"][0, class_idx]
    score.backward()
    attr = (x.grad.detach() * x.detach()).squeeze(0).cpu().numpy()
    return attr

def plot_attribution(x, attr, title, path):
    fig, axes = plt.subplots(12, 1, figsize=(12, 10), sharex=True)
    t = np.arange(x.shape[1]) / 100.0
    scale = np.max(np.abs(attr)) + 1e-8
    for i, ax in enumerate(axes):
        ax.plot(t, x[i], color="black", lw=0.6)
        ax.fill_between(t, 0, attr[i] / scale, color="tab:red", alpha=0.35)
        ax.set_ylabel(LEADS[i], rotation=0, labelpad=18)
    axes[0].set_title(title)
    axes[-1].set_xlabel("Tiempo (s)")
    plt.tight_layout()
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)

def run_attributions():
    model = load_best_model("P1_MTL_BALANCED_PROPOSED")
    cases = select_explainability_cases()
    for _, row in cases.iterrows():
        idx = int(row["global_idx"])
        x = read_ecg_record(resolve_record_stem(master.iloc[idx]))
        y = Y_HARD[idx]
        class_idx = int(np.argmax(y)) if y.sum() > 0 else 0
        diag_attr = grad_x_input(model, torch.tensor(x, dtype=torch.float32), target="diag", class_idx=class_idx)
        rev_attr = grad_x_input(model, torch.tensor(x, dtype=torch.float32), target="review")
        cid = f"{int(row['ecg_id'])}_{row['case_type']}"
        np.save(ATTR_DIR / f"{cid}_diag.npy", diag_attr)
        np.save(ATTR_DIR / f"{cid}_review.npy", rev_attr)
        plot_attribution(x, diag_attr, f"{cid} diag {CLASSES[class_idx]}", FIGURES_DIR / f"explainability_case_{cid}_diag.png")
        plot_attribution(x, rev_attr, f"{cid} review", FIGURES_DIR / f"explainability_case_{cid}_review.png")

def run_lead_ablation():
    model = load_best_model("P1_MTL_BALANCED_PROPOSED")
    sample = np.random.default_rng(SEED).choice(idx_test, size=min(LEAD_ABLATION_N, len(idx_test)), replace=False)
    rows = []
    for idx in tqdm(sample, desc="Lead ablation"):
        x = read_ecg_record(resolve_record_stem(master.iloc[idx]))
        with torch.no_grad():
            out = model(torch.tensor(x, dtype=torch.float32, device=DEVICE).unsqueeze(0))
            p0 = torch.sigmoid(out["diag_logits"]).cpu().numpy()[0]
            r0 = torch.sigmoid(out["review_logit"]).cpu().numpy()[0, 0]
        for lead_idx, lead in enumerate(LEADS):
            xa = x.copy()
            xa[lead_idx, :] = 0.0
            with torch.no_grad():
                out = model(torch.tensor(xa, dtype=torch.float32, device=DEVICE).unsqueeze(0))
                p1 = torch.sigmoid(out["diag_logits"]).cpu().numpy()[0]
                r1 = torch.sigmoid(out["review_logit"]).cpu().numpy()[0, 0]
            for k, cls in enumerate(CLASSES):
                rows.append({
                    "ecg_id": master.iloc[idx]["ecg_id"], "lead": lead, "class": cls,
                    "p_original": p0[k], "p_ablated": p1[k], "delta_p": p1[k] - p0[k],
                    "review_original": r0, "review_ablated": r1, "delta_review": r1 - r0,
                    "y_true": Y_HARD[idx, k], "ambiguity_score": AMBIGUITY[idx],
                    "review_binary": int(REVIEW_BINARY[idx]),
                })
    out = pd.DataFrame(rows)
    out.to_csv(EXPLAIN_DIR / "lead_ablation.csv", index=False)
    out.to_csv(FIGURE_DATA_DIR / "fig06_lead_ablation.csv", index=False)
    return out

def perturb(x, kind):
    rng = np.random.default_rng()
    z = x.copy()
    if kind == "gaussian_noise":
        z += rng.normal(0, 0.05, size=z.shape).astype(np.float32)
    elif kind == "baseline_wander":
        t = np.linspace(0, 10, z.shape[1])
        z += (0.15 * np.sin(2 * np.pi * 0.3 * t))[None, :]
    elif kind == "temporal_dropout":
        start = rng.integers(0, z.shape[1] - 100)
        z[:, start:start+100] = 0
    elif kind == "lead_dropout":
        z[rng.integers(0, 12), :] = 0
    return z.astype(np.float32)

def run_noise_perturbation():
    model = load_best_model("P1_MTL_BALANCED_PROPOSED")
    sample = np.random.default_rng(SEED).choice(idx_test, size=min(NOISE_PERTURB_N, len(idx_test)), replace=False)
    kinds = ["gaussian_noise", "baseline_wander", "temporal_dropout", "lead_dropout"]
    rows = []
    for idx in tqdm(sample, desc="Perturbaciones"):
        x = read_ecg_record(resolve_record_stem(master.iloc[idx]))
        with torch.no_grad():
            out = model(torch.tensor(x, dtype=torch.float32, device=DEVICE).unsqueeze(0))
            r0 = torch.sigmoid(out["review_logit"]).cpu().numpy()[0, 0]
        for kind in kinds:
            xp = perturb(x, kind)
            with torch.no_grad():
                out = model(torch.tensor(xp, dtype=torch.float32, device=DEVICE).unsqueeze(0))
                r1 = torch.sigmoid(out["review_logit"]).cpu().numpy()[0, 0]
            rows.append({"ecg_id": master.iloc[idx]["ecg_id"], "perturbation": kind, "review_original": r0, "review_perturbed": r1, "delta_review": r1 - r0})
    out = pd.DataFrame(rows)
    out.to_csv(EXPLAIN_DIR / "noise_perturbation.csv", index=False)
    return out

try:
    run_attributions()
    lead_ablation = run_lead_ablation()
    noise_perturb = run_noise_perturbation()
except Exception as e:
    log(f"Explicabilidad omitida por error recuperable: {e}")
    lead_ablation = pd.DataFrame()
    noise_perturb = pd.DataFrame()


In [ ]:
# ============================================================
# 21. Figuras principales y data fuente
# ============================================================
sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 130})

def savefig(name):
    path = FIGURES_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()
    return path

# Figura 1: pipeline.
for model_name in MODEL_ORDER:
    hist_path = MODELS_DIR / model_name / "history.csv"
    if hist_path.exists():
        hist = pd.read_csv(hist_path)
        hist.to_csv(FIGURE_DATA_DIR / f"training_history_{model_name}.csv", index=False)
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))
        axes[0].plot(hist["epoch"], hist["train_loss"], marker="o", label="train_loss")
        axes[0].set_title(f"{model_name} loss")
        axes[0].set_xlabel("epoch")
        axes[0].legend()
        if "val_diag_auprc_macro" in hist:
            axes[1].plot(hist["epoch"], hist["val_diag_auprc_macro"], marker="o", label="val_diag_AUPRC")
        if "val_review_auprc" in hist:
            axes[1].plot(hist["epoch"], hist["val_review_auprc"], marker="o", label="val_review_AUPRC")
        axes[1].set_title(f"{model_name} validation")
        axes[1].set_xlabel("epoch")
        axes[1].legend()
        savefig(f"training_curves_{model_name}.png")

# Figura 1: pipeline.
fig, ax = plt.subplots(figsize=(12, 4))
ax.axis("off")
boxes = [
    (0.05, 0.60, "ECG 12 derivaciones\nrecords100"),
    (0.32, 0.60, "Backbone CNN1D"),
    (0.62, 0.75, "Diagnostico multi-label\nNORM MI STTC CD HYP"),
    (0.62, 0.35, "Score de revision\nbaja certeza humana"),
    (0.05, 0.15, "Likelihoods PTB-XL"),
    (0.32, 0.15, "Ambiguedad humana\nmax(1-likelihood)"),
    (0.84, 0.35, "Aceptar apoyo automatico\n/ revision cuidadosa"),
]
for x, y, txt in boxes:
    ax.text(x, y, txt, ha="center", va="center", fontsize=11,
            bbox=dict(boxstyle="round,pad=0.35", fc="#f8fafc", ec="#334155"))
arrows = [((0.16,0.60),(0.25,0.60)), ((0.40,0.60),(0.55,0.75)), ((0.40,0.60),(0.55,0.35)),
          ((0.16,0.15),(0.25,0.15)), ((0.40,0.15),(0.55,0.35)), ((0.72,0.35),(0.78,0.35))]
for a,b in arrows:
    ax.annotate("", xy=b, xytext=a, arrowprops=dict(arrowstyle="->", lw=1.8))
savefig("fig01_pipeline.png")
pd.DataFrame({"node":[b[2] for b in boxes]}).to_csv(FIGURE_DATA_DIR / "fig01_pipeline_data.csv", index=False)

# Figura 2: ambiguedad y bins.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(master["ambiguity_score"], bins=30, ax=axes[0], color="#2563eb")
axes[0].set_title("Distribucion de ambiguedad humana")
axes[0].set_xlabel("a_i = max(1 - likelihood)")
bin_counts = master["ambiguity_bin"].value_counts().reindex(["B0","B1","B2","B3"]).fillna(0)
sns.barplot(x=bin_counts.index, y=bin_counts.values, ax=axes[1], color="#0f766e")
axes[1].set_title("Bins de ambiguedad")
axes[1].set_ylabel("N")
savefig("fig02_likelihood_ambiguity_distribution.png")

# Figura 3: rendimiento diagnostico.
plot_diag = diag_global[(diag_global["split"]=="test")][["model","AUROC_macro_thrval","AUPRC_macro_thrval","F1_macro_thrval"]].melt("model", var_name="metric", value_name="value")
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=plot_diag, x="metric", y="value", hue="model", ax=ax)
ax.set_ylim(0, 1)
ax.set_title("Rendimiento diagnostico en test")
ax.tick_params(axis="x", rotation=20)
savefig("fig03_diagnostic_performance.png")

# Figura 4: riesgo-cobertura.
risk_rows = []
for score_name, diag_name in score_to_diag.items():
    for q in Q_VALUES:
        row = selective_table[(selective_table["model_or_score"]==score_name) & (selective_table["q"]==q)].iloc[0]
        risk_rows.append({"model_or_score": score_name, "coverage": row["coverage"], "risk_1_minus_f1": 1-row["F1_accepted"]})
risk_df = pd.DataFrame(risk_rows)
risk_df.to_csv(FIGURE_DATA_DIR / "fig04_risk_coverage.csv", index=False)
fig, ax = plt.subplots(figsize=(8,5))
sns.lineplot(data=risk_df, x="coverage", y="risk_1_minus_f1", hue="model_or_score", marker="o", ax=ax)
ax.invert_xaxis()
ax.set_title("Curva riesgo-cobertura")
ax.set_ylabel("1 - F1 en ECG aceptados")
savefig("fig04_risk_coverage.png")

# Figura 5: captura de errores y baja certeza.
cap_df = selective_table.melt(
    id_vars=["model_or_score","q"], value_vars=["ErrorCapture@q","LowCertCapture@q"],
    var_name="capture_type", value_name="capture"
)
fig, ax = plt.subplots(figsize=(10,5))
sns.barplot(data=cap_df, x="q", y="capture", hue="model_or_score", ax=ax)
ax.set_ylim(0, 1)
ax.set_title("Captura de errores y baja certeza por fraccion derivada")
savefig("fig05_error_capture_at_q.png")

# Figura 6: explicabilidad agregada por lead.
if isinstance(lead_ablation, pd.DataFrame) and not lead_ablation.empty:
    lead_plot = lead_ablation.groupby("lead").agg(
        mean_abs_delta_p=("delta_p", lambda x: np.mean(np.abs(x))),
        mean_abs_delta_review=("delta_review", lambda x: np.mean(np.abs(x))),
    ).reset_index()
    order = LEADS
    lead_plot["lead"] = pd.Categorical(lead_plot["lead"], categories=order, ordered=True)
    lead_plot = lead_plot.sort_values("lead")
    fig, ax = plt.subplots(figsize=(10,5))
    lead_plot_m = lead_plot.melt("lead", var_name="effect", value_name="mean_abs_delta")
    sns.barplot(data=lead_plot_m, x="lead", y="mean_abs_delta", hue="effect", ax=ax)
    ax.set_title("Lead ablation: efecto diagnostico vs revision")
    savefig("fig06_explainability_lead_ablation.png")
else:
    fig, ax = plt.subplots(figsize=(8,3))
    ax.axis("off")
    ax.text(0.5,0.5,"Explicabilidad pendiente: ejecutar P1 y lead ablation", ha="center", va="center")
    savefig("fig06_explainability_lead_ablation.png")


In [ ]:
# ============================================================
# 22. Manifiestos de figuras y tablas
# ============================================================
fig_manifest = pd.DataFrame([
    {"figure_id":"fig01","figure_filename":"fig01_pipeline.png","source_data_file":"fig01_pipeline_data.csv","generated_by_cell_or_function":"cell_21","question_answered":"Narrativa del metodo","x_axis":"NA","y_axis":"NA","models_included":"all","interpretation_short":"Los likelihoods entrenan la derivacion, no reemplazan el diagnostico."},
    {"figure_id":"fig02","figure_filename":"fig02_likelihood_ambiguity_distribution.png","source_data_file":"fig02_likelihood_distribution.csv","generated_by_cell_or_function":"cell_21","question_answered":"P1","x_axis":"ambiguedad/bins","y_axis":"N","models_included":"none","interpretation_short":"Audita desbalance de certeza humana."},
    {"figure_id":"fig03","figure_filename":"fig03_diagnostic_performance.png","source_data_file":"fig03_diagnostic_metrics.csv","generated_by_cell_or_function":"cell_21","question_answered":"P2","x_axis":"metricas","y_axis":"valor","models_included":"B1 B2 P0 P1","interpretation_short":"Compara si P1 mantiene rendimiento diagnostico."},
    {"figure_id":"fig04","figure_filename":"fig04_risk_coverage.png","source_data_file":"fig04_risk_coverage.csv","generated_by_cell_or_function":"cell_21","question_answered":"P5","x_axis":"coverage","y_axis":"1-F1","models_included":"B3 P0 P1 Random","interpretation_short":"Evalua utilidad operacional al derivar casos."},
    {"figure_id":"fig05","figure_filename":"fig05_error_capture_at_q.png","source_data_file":"fig05_error_capture.csv","generated_by_cell_or_function":"cell_21","question_answered":"P5/P6","x_axis":"q","y_axis":"capture","models_included":"B3 P0 P1 Random","interpretation_short":"Cuantifica errores y baja certeza capturados por revision."},
    {"figure_id":"fig06","figure_filename":"fig06_explainability_lead_ablation.png","source_data_file":"fig06_lead_ablation.csv","generated_by_cell_or_function":"cell_20/21","question_answered":"P7","x_axis":"lead","y_axis":"delta medio absoluto","models_included":"P1","interpretation_short":"Compara sensibilidad por derivacion de diagnostico y revision."},
])
fig_manifest.to_csv(MANIFEST_DIR / "figures_manifest.csv", index=False)

table_manifest = pd.DataFrame([
    {"table_id":"tab01","table_filename":"dataset_audit.csv","question_answered":"P1","rows_description":"splits train/val/test","columns_description":"N, prevalencias, bins de ambiguedad","interpretation_short":"Base para justificar balanceo."},
    {"table_id":"tab02","table_filename":"diagnostic_metrics_global.csv","question_answered":"P2","rows_description":"modelos por split","columns_description":"AUROC, AUPRC, F1, Brier, ECE","interpretation_short":"Compara rendimiento diagnostico."},
    {"table_id":"tab03","table_filename":"review_metrics.csv","question_answered":"P3/P4","rows_description":"scores de revision","columns_description":"AUROC/AUPRC revision, MAE, RMSE, Spearman","interpretation_short":"Evalua aprendizaje de baja certeza humana."},
    {"table_id":"tab04","table_filename":"selective_referral_metrics.csv","question_answered":"P5","rows_description":"score por q","columns_description":"coverage, ErrorCapture, LowCertCapture","interpretation_short":"Evalua utilidad operacional de derivacion."},
    {"table_id":"tab05","table_filename":"balance_effect.csv","question_answered":"P4","rows_description":"P0 y P1","columns_description":"review_AUPRC, MAE por bin, ErrorCapture@20, LowCertCapture@20","interpretation_short":"Evalua si el balanceo por bins mejora revision."},
    {"table_id":"tab06","table_filename":"selective_referral_by_class.csv","question_answered":"P6","rows_description":"score, q, clase","columns_description":"captura de errores, FN y FP","interpretation_short":"Analiza clases dificiles como MI e HYP."},
    {"table_id":"tab07","table_filename":"subgroup_metrics.csv","question_answered":"Subgrupos","rows_description":"modelo por subgrupo","columns_description":"N, diagnostico, revision, ErrorCapture","interpretation_short":"Analisis descriptivo por sexo y edad."},
    {"table_id":"tab08","table_filename":"bootstrap_summary.csv","question_answered":"Incertidumbre estadistica","rows_description":"comparaciones pareadas","columns_description":"media e IC95","interpretation_short":"Cuantifica incertidumbre de diferencias clave."},
    {"table_id":"tab09","table_filename":"master_predictions_test.csv","question_answered":"Reanalisis posterior","rows_description":"ECG test","columns_description":"labels, probabilidades, scores, derivaciones","interpretation_short":"Archivo maestro para responder preguntas sin reentrenar."},
])
table_manifest.to_csv(MANIFEST_DIR / "tables_manifest.csv", index=False)


In [ ]:
# ============================================================
# 23. REPORT_SUMMARY y NEXT_STEPS
# ============================================================
def first_row_value(df, filt, col):
    sub = df.query(filt)
    if len(sub) == 0 or col not in sub:
        return "NA"
    v = sub.iloc[0][col]
    return "NA" if pd.isna(v) else f"{v:.3f}" if isinstance(v, (float, np.floating)) else str(v)

summary = f'''# HCG-SelectiveECG - Report Summary

## Objetivo

Evaluar si los likelihoods diagnosticos de PTB-XL pueden usarse como senal
auxiliar de certeza humana para derivacion selectiva a revision medica, sin
convertirlos necesariamente en targets diagnosticos suaves.

## Dataset y splits

- Dataset: PTB-XL records100.
- Clases: {", ".join(CLASSES)}.
- Train: folds 1-8, N={len(idx_train)}.
- Validation: fold 9, N={len(idx_val)}.
- Test: fold 10, N={len(idx_test)}.

## Modelos comparados

- B1_HARD_DIAG: diagnostico hard-label.
- B2_SOFT_DIRECT: likelihood/100 como target diagnostico directo.
- B3_THRESHOLD_UNCERTAINTY: score de revision por cercania al umbral validado de B1.
- P0_MTL_UNBALANCED: diagnostico + revision sin balanceo.
- P1_MTL_BALANCED_PROPOSED: diagnostico + revision con balanceo por bins.

## Construccion de ambiguedad humana

Para cada ECG se calcula `a_i = max(1 - likelihood_k)` entre clases positivas.
La etiqueta de revision es `r_i = 1` si `a_i >= {ETA_REVIEW}`.

## Desbalance y balanceo

Pesos por bins calculados solo en train:

```json
{json.dumps(BIN_WEIGHTS, indent=2)}
```

## Resultados diagnosticos principales

Ver `results/tables/diagnostic_metrics_global.csv`.

## Resultados de revision medica

Ver `results/tables/review_metrics.csv`.

## Derivacion selectiva

Ver `results/tables/selective_referral_metrics.csv` y
`results/tables/selective_referral_by_class.csv`.

## Bootstrap

Ver `results/tables/bootstrap_summary.csv`.

## Explicabilidad

Ver `results/explainability/`, `results/figures/fig06_explainability_lead_ablation.png`
y figuras por caso si se pudieron generar.

## Limitaciones

- Likelihood es un proxy imperfecto de certeza humana.
- Validacion interna en PTB-XL; falta validacion externa.
- Modelo CNN1D liviano.
- En debug los resultados no son reportables.
- Si no hay `patient_id`, bootstrap se hace por ECG.

## Rutas

- RUN_DIR: `{RUN_DIR}`
- Master test: `{TABLES_DIR / "master_predictions_test.csv"}`
- Figuras: `{FIGURES_DIR}`
- Tablas: `{TABLES_DIR}`
'''

(RUN_DIR / "REPORT_SUMMARY.md").write_text(summary, encoding="utf-8")

next_steps = '''# Next Steps for Paper

1. Usar Figura 1 para explicar que el metodo no reemplaza diagnostico: aprende derivacion.
2. Usar Tabla 3 y Tabla 4 para sostener la contribucion principal: revision y derivacion selectiva.
3. Reportar Tabla 2 solo como condicion de seguridad: P1 debe mantener diagnostico comparable.
4. Si P1 no supera B3, presentarlo como resultado negativo util y reforzar analisis de umbrales.
5. Si P1 supera B3 en ErrorCapture@20% o review AUPRC, centrar el resumen en utilidad operacional.
6. Para revista, repetir con multiples semillas, records500, arquitectura mas fuerte y validacion externa.
7. Revisar manualmente casos de MI/HYP derivados y falsos negativos capturados.
'''
(RUN_DIR / "NEXT_STEPS_FOR_PAPER.md").write_text(next_steps, encoding="utf-8")

run_summary = {
    "run_id": RUN_ID,
    "run_mode": RUN_MODE,
    "project_root": str(PROJECT_ROOT),
    "run_dir": str(RUN_DIR),
    "device": str(DEVICE),
    "gpu_name": GPU_NAME,
    "classes": CLASSES,
    "paths": {"raw_base": str(RAW_BASE), "records100": str(RECORDS100_PATH)},
    "splits": {"train": int(len(idx_train)), "val": int(len(idx_val)), "test": int(len(idx_test))},
    "models": MODEL_ORDER,
    "outputs": {
        "tables": str(TABLES_DIR),
        "figures": str(FIGURES_DIR),
        "master_test": str(TABLES_DIR / "master_predictions_test.csv"),
        "report_summary": str(RUN_DIR / "REPORT_SUMMARY.md"),
    },
    "warnings": [
        "No interpretar resultados debug como finales.",
        "Analisis de subgrupos es descriptivo, no conclusion de equidad clinica.",
    ],
}
save_json(run_summary, RUN_DIR / "RUN_SUMMARY.json")

print("RUN COMPLETADO")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUN_DIR:", RUN_DIR)
print("FIGURES:", FIGURES_DIR)
print("TABLES:", TABLES_DIR)
print("MASTER TEST FILE:", TABLES_DIR / "master_predictions_test.csv")
print("REPORT SUMMARY:", RUN_DIR / "REPORT_SUMMARY.md")


In [ ]:
# ============================================================
# 24. Paquete opcional de resultados liviano
# ============================================================
import zipfile

DELIVERABLES_DIR.mkdir(parents=True, exist_ok=True)
zip_path = DELIVERABLES_DIR / f"ptbxl_hcg_selective_results_{RUN_ID}.zip"

include_dirs = [CONFIG_DIR, TABLES_DIR, FIGURE_DATA_DIR, FIGURES_DIR, EXPLAIN_DIR, MANIFEST_DIR, LOGS_DIR]
include_files = [RUN_DIR / "RUN_SUMMARY.json", RUN_DIR / "REPORT_SUMMARY.md", RUN_DIR / "NEXT_STEPS_FOR_PAPER.md"]
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for d in include_dirs:
        for p in d.rglob("*"):
            if p.is_file():
                z.write(p, p.relative_to(RUN_DIR).as_posix())
    for p in include_files:
        if p.exists():
            z.write(p, p.relative_to(RUN_DIR).as_posix())

print("ZIP liviano:", zip_path)


## 25. Agregacion multi-semilla (media +/- DE entre semillas)

Ejecutar SOLO despues de haber corrido las semillas de `SEED_LIST`.
Reutiliza la corrida original (seed 42) via `SEED_TO_RUNID` y combina
todas las semillas disponibles. Produce, para diagnostico, revision,
derivacion selectiva y las diferencias pareadas del bootstrap:
media, desviacion estandar e IC95% (t de Student) entre semillas.
Un efecto es robusto si su IC95% entre semillas no cruza 0.


In [ ]:
# ============================================================
# 25. Agregacion multi-semilla (robustez estadistica para el paper)
# ============================================================
# Ejecutar SOLO despues de correr las semillas de SEED_LIST.
# Reutiliza la corrida ya completada (seed 42) via SEED_TO_RUNID.
from scipy import stats as _stats

AGG_DIR = RUNS_DIR / "_aggregate_multiseed"
AGG_DIR.mkdir(parents=True, exist_ok=True)

# Mapeo semilla -> RUN_ID. Por defecto nombre estable; override para la corrida original.
SEED_TO_RUNID = {seed: f"hcg_selective_ecg_{RUN_MODE}_seed{seed}" for seed in SEED_LIST}
SEED_TO_RUNID[42] = "hcg_selective_ecg_full_20260712_131520"  # corrida original ya completada

def _seed_tables_dir(seed):
    return RUNS_DIR / SEED_TO_RUNID[seed] / "results" / "tables"

available_seeds = [s for s in SEED_LIST if _seed_tables_dir(s).exists()]
missing_seeds = [s for s in SEED_LIST if s not in available_seeds]
print("Semillas disponibles para agregar:", available_seeds)
if missing_seeds:
    print("Semillas faltantes (aun no corridas):", missing_seeds)

def _read_seed_table(seed, name):
    p = _seed_tables_dir(seed) / name
    if not p.exists():
        return None
    df = pd.read_csv(p)
    df["seed"] = seed
    return df

def _concat_tables(name):
    parts = [t for t in (_read_seed_table(s, name) for s in available_seeds) if t is not None]
    return pd.concat(parts, ignore_index=True) if parts else None

def _agg_mean_sd(df, group_cols, value_cols):
    rows = []
    for keys, g in df.groupby(group_cols, dropna=False):
        keys = keys if isinstance(keys, tuple) else (keys,)
        row = dict(zip(group_cols, keys))
        for v in value_cols:
            vals = pd.to_numeric(g[v], errors="coerce").dropna().values
            row["n_seeds"] = int(len(vals))
            row[f"{v}_mean"] = float(np.mean(vals)) if len(vals) else np.nan
            row[f"{v}_sd"] = float(np.std(vals, ddof=1)) if len(vals) > 1 else np.nan
            if len(vals) > 1:
                se = np.std(vals, ddof=1) / np.sqrt(len(vals))
                tcrit = float(_stats.t.ppf(0.975, len(vals) - 1))
                row[f"{v}_ci95_low"] = float(np.mean(vals) - tcrit * se)
                row[f"{v}_ci95_high"] = float(np.mean(vals) + tcrit * se)
            else:
                row[f"{v}_ci95_low"] = np.nan
                row[f"{v}_ci95_high"] = np.nan
        rows.append(row)
    return pd.DataFrame(rows)

summaries = {}

if len(available_seeds) == 0:
    print("No hay corridas disponibles; corre al menos una semilla antes de agregar.")
else:
    # --- Diagnostico (solo test) ---
    diag = _concat_tables("diagnostic_metrics_global.csv")
    if diag is not None:
        diag_test = diag[diag["split"] == "test"].copy()
        diag_agg = _agg_mean_sd(
            diag_test, ["model"],
            ["AUROC_macro_thrval", "AUPRC_macro_thrval", "F1_macro_thrval",
             "Brier_macro_thrval", "ECE_macro_thrval"],
        )
        diag_agg.to_csv(AGG_DIR / "multiseed_diagnostic_global_test.csv", index=False)
        summaries["diagnostic"] = diag_agg

    # --- Revision medica (test) ---
    rev = _concat_tables("review_metrics.csv")
    if rev is not None:
        rev_test = rev[rev["split"] == "test"].copy() if "split" in rev.columns else rev
        rev_agg = _agg_mean_sd(
            rev_test, ["model_or_score"],
            ["review_AUROC", "review_AUPRC", "review_F1",
             "MAE_ambiguity", "Spearman_ambiguity"],
        )
        rev_agg.to_csv(AGG_DIR / "multiseed_review_metrics.csv", index=False)
        summaries["review"] = rev_agg

    # --- Derivacion selectiva en q=0.20 ---
    sel = _concat_tables("selective_referral_metrics.csv")
    if sel is not None:
        sel20 = sel[np.isclose(sel["q"], 0.20)].copy()
        sel_agg = _agg_mean_sd(
            sel20, ["model_or_score"],
            ["ErrorCapture@q", "LowCertCapture@q", "F1_accepted",
             "Brier_accepted", "ECE_accepted"],
        )
        sel_agg.to_csv(AGG_DIR / "multiseed_selective_referral_q20.csv", index=False)
        summaries["selective_q20"] = sel_agg

    # --- Diferencias pareadas (bootstrap por semilla -> agregacion entre semillas) ---
    boot = _concat_tables("bootstrap_summary.csv")
    if boot is not None:
        boot_agg = _agg_mean_sd(boot, ["comparison"], ["mean"]).rename(columns={
            "mean_mean": "diff_mean", "mean_sd": "diff_sd",
            "mean_ci95_low": "diff_ci95_low", "mean_ci95_high": "diff_ci95_high",
        })
        boot_agg["significativo_entre_semillas"] = (
            (boot_agg["diff_ci95_low"] > 0) | (boot_agg["diff_ci95_high"] < 0)
        )
        boot_agg.to_csv(AGG_DIR / "multiseed_paired_diffs.csv", index=False)
        summaries["paired_diffs"] = boot_agg

    # --- Resumen legible en Markdown ---
    def _fmt(df):
        d = df.round(4)
        try:
            return d.to_markdown(index=False)
        except Exception:
            return d.to_string(index=False)

    lines = []
    lines.append("# MULTISEED SUMMARY - HCG-SelectiveECG\n")
    lines.append(f"- Semillas agregadas: {available_seeds}")
    lines.append(f"- Semillas faltantes: {missing_seeds}")
    lines.append(f"- n_semillas efectivas: {len(available_seeds)}\n")
    lines.append("Un efecto se considera robusto si su IC95% entre semillas no cruza 0.\n")
    if "paired_diffs" in summaries:
        lines.append("## Diferencias pareadas (media entre semillas)\n")
        lines.append(_fmt(summaries["paired_diffs"]))
        lines.append("")
    if "review" in summaries:
        lines.append("## Revision medica (test)\n")
        lines.append(_fmt(summaries["review"]))
        lines.append("")
    if "selective_q20" in summaries:
        lines.append("## Derivacion selectiva @ q=0.20\n")
        lines.append(_fmt(summaries["selective_q20"]))
        lines.append("")
    if "diagnostic" in summaries:
        lines.append("## Diagnostico (test)\n")
        lines.append(_fmt(summaries["diagnostic"]))
        lines.append("")
    (AGG_DIR / "MULTISEED_SUMMARY.md").write_text("\n".join(lines), encoding="utf-8")

    print("Agregacion multi-semilla escrita en:", AGG_DIR)
    if "paired_diffs" in summaries:
        print("\nDiferencias pareadas entre semillas:")
        cols = ["comparison", "diff_mean", "diff_sd",
                "diff_ci95_low", "diff_ci95_high", "significativo_entre_semillas"]
        print(summaries["paired_diffs"][cols].to_string(index=False))


## Qué debe devolver Ronald

Después de correr, debe enviar de vuelta preferentemente la carpeta completa:

```text
ptbxl_likelihood_calibration/runs_hcg_selective/RUN_ID/
```

Si quiere enviar algo liviano, el notebook también crea:

```text
ptbxl_likelihood_calibration/deliverables_hcg_selective/ptbxl_hcg_selective_results_RUN_ID.zip
```

Para análisis/redacción, con ese ZIP liviano basta en la mayoría de los casos. Para reanudar entrenamiento o revisar checkpoints, hace falta la carpeta completa del `RUN_ID`.
